# Sales Brief — Databricks Job 執行步驟（v2：雙模板＋chart JSON 三欄交付）

> **改版目的**: 下游 myAgent 寄信時，信件內文（send_mail body）有體積上限，且 Outlook
> 不支援 SVG——先前由 LLM 把完整 HTML 轉成 Email 格式的做法造成約 2.2x 體積膨脹而超限。
> 因此改為 **job 端直接多產兩個欄位**：預製好的 Email 內文（`mail_body`）與圖表數據
> JSON（`chart_data`），myAgent 只負責畫 PNG、替換佔位符、寄信，不再做任何格式轉換。
>
> **改版範圍**：現有 job 的取數、判斷、SVG 產圖、MERGE 寫入**全部沿用不動**；
> 只在既有迴圈內新增 3b / 3c 兩個產出與對應驗證。標【既有】＝已實作照舊、
> 標【新增】＝本次要加的。

---

## 交付物與分工（一張表看懂）

| 產出 | 產生者 | 目標表欄位 | 說明 |
|------|--------|-----------|------|
| 完整圖表版 HTML（附件） | job【既有】 | `html_content` | 現有 Step 3 產出：`ATTACHMENT_TEMPLATE` 填實際數據＋SVG 折線圖＋AI Insight；收件人用瀏覽器開啟，隨信 .html 附件 |
| Email 內文 HTML | job【新增】 | `mail_body` | `MAIL_BODY_TEMPLATE` 填實際數據（640px table 版型、minified），圖表處保留 `{{CHART1_PNG_URL}}` / `{{CHART2_PNG_URL}}` 佔位符 |
| 圖表數據 JSON | job【新增】 | `chart_data` | 2 組圖表數據組在**同一個 JSON**（`charts` 陣列），供 myAgent 逐一畫成靜態 PNG；與 SVG 產圖共用同一份序列（同源） |
| 靜態 PNG ＋ 寄信 | myAgent | — | 依 `chart_data` 畫圖 → 取得 https URL → 替換 `mail_body` 佔位符 → `html_content` 落地當附件 → 寄出 |

---

## 執行步驟

### 環境與參考（＝現有 job，不變）

- 來源數據（三張，皆在 `micenter.mi3_datahub_prod`）：
  `g_sales_brief_numeric_entry`、`g_sales_brief_brand_summary`、`g_sales_brief_app_summary`
- LLM API 範例：`/jobs/00_sandbox/AzureFoundry_Claude_LLM_API`；
  Prompt 參考：`/jobs/sales_brief/sales_brief_summary`

### Step 0: 安裝套件【既有】

- `pip install -U anthropic httpx` + restartPython

### Step 1: 讀取來源 & 判斷各應用是否需要產出【既有，不變】

- 查三張來源表各 app 的最新 `fill_year * 100 + fill_month`
- 取三表的 `LEAST` 作為該 app 可用的最新完整月份
- 對 TV / NB / MNT 逐一判斷：有完整月份即產出
- 每次執行都會重新產出所有 app 最新月份（**非增量判斷**；重複執行靠 Step 4 MERGE 冪等）

### Step 2: 資料整理函式【既有＋新增】

- 【既有】品牌排序、數據格式化、AI Insight 生成等 helper functions；
  LLM fallback placeholder（尚未初始化時回傳空值）
- 【既有】季度數據由 `get_quarterly_data()` 從月度值換算（`numeric_entry` 無季度 metric）：
  撈 `category='panel_buy' AND metric='monthly'` → 每 `(brand_name, target_year, target_month)`
  以 `ROW_NUMBER`（`fill_year DESC, fill_month DESC`）取最新提交版 →
  `q = (tm - 1) // 3 + 1` 分季 → 同季三個月 SUM；QoQ% 亦於函式內對前一季計算
- 【新增】月度取數規則——以該品牌實際有值月份為準，僅當來源亦無資料時顯示「-」
  （修正前版「取數區間截斷」造成 Aug/Sep 空格的問題；若既有格式化函式已如此則免改）
- 【新增】抽出兩組圖表的原始序列為**獨立中間結構**（dict / dataclass），
  供 3a 的 SVG 產圖與 3c 的 chart JSON 共用，確保附件 SVG 與信件 PNG **同源**：
  - Chart 1：品牌月度面板採購趨勢（月份 × 4 品牌，含 Forecast 分界索引）
  - Chart 2：主要品牌庫存週數趨勢（月份 × 4 品牌）

### Step 2b: LLM 初始化【既有，不變】

- 透過 Azure Foundry 呼叫 Claude（claude-sonnet-5）產各 Section AI Insight
  （Section 1–5 的 insight / 摘要 / 判讀現況均已涵蓋，不需擴充）
- 原則不變：LLM 只產**分析文字**，不碰數字表格與 HTML 結構（數字一律由程式填入）
- 【新增注意】LLM fallback 回空值時，`mail_body` 的 insight 區塊仍須結構完整
  （空 insight 顯示為空框或整塊省略擇一定案，不得產出殘破 HTML）

### Step 3: HTML / mail_body / chart JSON 產出函式【既有＋新增 3b、3c】

**模板皆為 notebook 內變數**：
- `ATTACHMENT_TEMPLATE`【既有變數改名】：附件版 HTML 模板（原 TEMPLATE 變數，改名屬一次性調整）
- `MAIL_BODY_TEMPLATE`【已存入】：信件內文模板（內容 = `sales_brief_email_template.html`）

**3a｜附件版 `html_content`【既有，不變】**
- 依 `ATTACHMENT_TEMPLATE` 結構填入各 app 實際數據，含 SVG 折線圖＋各 Section AI Insight
- 每個 app × 每個月份 = 一個 HTML
- 唯一調整：SVG 產圖的輸入改吃 Step 2 的圖表中間結構（與 3c 同源），繪圖邏輯不動

**3b｜信件內文 `mail_body`【新增】**（來源模板：變數 `MAIL_BODY_TEMPLATE`）
- 同一份數據填入 email 模板（table-based、head `<style>` class 化、寬 640px）
- 圖表區塊**保留** `{{CHART1_PNG_URL}}` / `{{CHART2_PNG_URL}}` 佔位符，其餘不留任何 `{{…}}`
- 輸出 minified（去縮排與多餘空白），控制體積

**3c｜圖表數據 `chart_data`【新增】**（2 組圖表組在同一個 JSON）

```json
{
  "app_name": "MNT",
  "brief_month": "2026-07",
  "charts": [
    {
      "id": "CHART1",
      "placeholder": "{{CHART1_PNG_URL}}",
      "chart_type": "line",
      "title": "四品牌月度面板採購趨勢（MI3.0）",
      "unit": "M pcs",
      "x_labels": ["25-Jan", "Feb", "...", "Sep(F)"],
      "forecast_from_index": 17,
      "series": [
        { "name": "Dell",    "color": "#2a78d6", "data": [2.4, 1.5, "..."] },
        { "name": "Samsung", "color": "#eb6834", "data": ["..."] },
        { "name": "HP",      "color": "#1baf7a", "data": ["..."] },
        { "name": "ASUS",    "color": "#eda100", "data": ["..."] }
      ]
    },
    {
      "id": "CHART2",
      "placeholder": "{{CHART2_PNG_URL}}",
      "chart_type": "line",
      "title": "主要品牌庫存週數趨勢（MI3.0）",
      "unit": "week",
      "x_labels": ["25-07", "...", "26-07"],
      "forecast_from_index": null,
      "series": [ "…同上結構…" ]
    }
  ]
}
```

- schema 約定（job 與 myAgent 的**唯一契約**，兩端不得各自解讀）：
  - `charts` 陣列長度 = 信件內佔位符數量 = myAgent 應產出的 PNG 張數
  - `placeholder` 逐字對應 `mail_body` 內的佔位符字串
  - `forecast_from_index`：自該索引（含）之後為預測值（虛線呈現）；無預測段為 `null`
  - 色碼沿用既有 SVG 色盤（Dell `#2a78d6` / Samsung `#eb6834` / HP `#1baf7a` /
    ASUS `#eda100`），附件 SVG 與信件 PNG 同色系
  - 缺值以 `null` 表示，畫圖時跳點、不補 0

### Step 4: 執行產出 & MERGE 寫入 Delta Table【既有＋加兩欄】

- 【既有】迴圈產出所有 app 的結果收集至 results list；
  以 `(app_name, brief_month)` 為 key 執行 MERGE：已存在 → UPDATE、不存在 → INSERT；
  歷史月份資料保留不刪除
- 目標表 `micenter.mi3_datahub_prod.g_sales_brief_mail_report`，
  Schema（前四欄＝既有；後兩欄**新增**，以 schema evolution 加欄，既有欄不動）：
  - `app_name` (STRING)：TV / NB / MNT
  - `brief_month` (STRING)：e.g. "2026-07"
  - `html_content` (STRING)：完整圖表版 HTML（附件用，含 SVG）
  - `created_at` (TIMESTAMP)：產出時間
  - `mail_body` (STRING)【新增】：Email 內文 HTML（含 `{{CHARTn_PNG_URL}}` 佔位符、minified）
  - `chart_data` (STRING)【新增】：圖表數據 JSON（見 3c schema）
- MERGE 的 UPDATE 子句同步更新新增兩欄
- **⚠ 一次性動作與例行 job 分離**：加欄（`ALTER TABLE … ADD COLUMNS (mail_body STRING, chart_data STRING)`）
  等 schema 增修屬**一次性 migration**，單獨手動執行一次即可，**不要寫進 job 讓每次執行都跑**；
  job 本體只做 MERGE，假設 schema 已就緒

### Step 5: 驗證結果 & HTML 預覽【既有＋新增關卡】

- 【既有】讀取目標表確認寫入正確、顯示 HTML 預覽
- 【新增】寫入前品質關卡：

| 驗證項 | 規則 | 目的 |
|--------|------|------|
| 佔位符完整性 | `mail_body` 恰含 `{{CHART1_PNG_URL}}`、`{{CHART2_PNG_URL}}` 各一次，無其他 `{{…}}` 殘留；`html_content` 不含任何 `{{…}}` | 防模板填漏 |
| 體積預算 | `mail_body` ≤ 35,000 字元（下游 send_mail 上限的保守值，實測定位後改為上限×0.9） | 防超過寄信內文上限 |
| chart JSON 可解析 | `json.loads` 成功；每組 `series[].data` 長度 = `x_labels` 長度；`charts` 數 = `mail_body` 佔位符數 | 防兩端契約漂移 |
| HTML 結構 | `html_content` 與 `mail_body` 均以 `</html>` 結尾；`mail_body` 無 `<svg>`、無 `<script>` | Email 相容底線（Outlook 不渲染 SVG/JS） |
| insight 完整性 | LLM fallback（空值）時 `mail_body` / `html_content` 結構仍完整、無殘破區塊 | 防半成品寄出 |

---

## 下游使用方式（myAgent 契約摘要——說明三欄怎麼被消費，job 端不需實作）

1. 查表取指定（或最新）`app_name` × `brief_month` 的三欄
2. 解析 `chart_data.charts`，每組畫一張靜態 PNG，取得 https URL
3. 將 `mail_body` 中各 `placeholder` 逐字替換為對應 PNG URL（唯一的編輯動作）；
   `html_content` 逐字落地為 .html 附件
4. 驗證（無殘留 `{{`、img 數 = charts 數）後寄出

> 因此 job 端三欄都必須是「**開箱即用的最終版**」：`mail_body` 除佔位符外不留任何
> 待處理標記；`chart_data` 自帶畫圖所需的全部資訊，下游不回查來源表。

---

## 注意事項

| # | 事項 | 說明 |
|---|------|------|
| 1 | `mail_body` 體積是硬約束 | 下游寄信內文有上限（確切值待實測）；job 端以 35k 保守值把關，超標視為產出失敗 |
| 2 | chart JSON schema 以本文件 3c 為 SoT | job 改 schema 必須先改本文件，再通知下游同步，防契約漂移 |
| 3 | 非增量重產的併發窗口 | job 每次重產最新月，若執行中恰被下游取數，可能拿到新舊混合欄位；建議單一 MERGE 交易內同時更新三欄（同列原子更新即無此問題） |
| 4 | 一次性動作不進例行 job | schema 加欄、`pip install` 以外的環境調整、歷史資料回填等 migration 行為，一律單獨執行一次，不寫進排程 job 每次跑 |

## 驗收清單

- [ ] MNT 單應用產出三欄，通過 Step 5 全部驗證關卡
- [ ] `mail_body` 以瀏覽器預覽：640px 版型正確、數據與 `html_content` 一致、佔位符位置正確
- [ ] `chart_data` 與 SVG 圖表數據逐值核對一致（同源驗證）
- [ ] 下游 myAgent 端對接寄信實測（由 myAgent 側執行，非本 job 範圍）

---

## 附錄：現有 job notebook README 原文（2026-08-24 抄錄，as-is 基準）

```markdown
# Sales Brief Mail Report 產出 Job

## 來源數據
- `micenter.mi3_datahub_prod.g_sales_brief_numeric_entry`
- `micenter.mi3_datahub_prod.g_sales_brief_brand_summary`
- `micenter.mi3_datahub_prod.g_sales_brief_app_summary`

## 目標表
- `micenter.mi3_datahub_prod.g_sales_brief_mail_report`
  - Schema: `app_name` (STRING), `brief_month` (STRING), `html_content` (STRING), `created_at` (TIMESTAMP)

## LLM 參考
- API 範例：`/jobs/00_sandbox/AzureFoundry_Claude_LLM_API`
- Prompt 參考：`/jobs/sales_brief/sales_brief_summary`

---

## 執行步驟

### Step 0: 安裝套件
- `pip install -U anthropic httpx` + restartPython

### Step 1: 讀取來源 & 判斷各應用是否需要產出
- 查三張來源表各 app 的最新 `fill_year * 100 + fill_month`
- 取三表的 `LEAST` 作為該 app 可用的最新完整月份
- 對 TV / NB / MNT 逐一判斷：有完整月份即產出
- 每次執行都會重新產出所有 app 最新月份（非增量判斷）

### Step 2: 資料整理函式
- 定義品牌排序、數據格式化、AI Insight 生成等 helper functions
- LLM fallback placeholder（尚未初始化時回傳空值）

### Step 2b: LLM 初始化
- 透過 Azure Foundry 呼叫 Claude (claude-sonnet-5)
- 啟用後 Section 1（年度/季度分析）與 Section 2（月度趨勢）皆會產生 AI Insight

### Step 3: HTML 產出函式
- 依 TEMPLATE 結構填入各 app 實際數據
- 含 SVG 折線圖 + 各 Section AI Insight
- 每個 app × 每個月份 = 一個 HTML

### Step 4: 執行產出 & MERGE 寫入 Delta Table
- 迴圈產出所有 app 的 HTML，收集至 results list
- 以 `(app_name, brief_month)` 為 key 執行 MERGE：
  - 已存在 → 更新 html_content + created_at
  - 不存在 → INSERT 新增
- 歷史月份資料保留不刪除

### Step 5: 驗證結果 & HTML 預覽
- 讀取目標表確認寫入正確
- 顯示 HTML 預覽
```

In [ ]:
ATTACHMENT_TEMPLATE = '''
<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>Jul'26 Monitor Sales Brief</title>
<!--
  Sales Brief HTML Template v2（web 完整版，含 MI3.0 折線圖）
  ─ 用途：瀏覽器開啟的完整報告頁（信件「點進去後露出」的落地頁），
    亦作為產圖流程的截圖來源（轉 PNG 後可嵌入 Outlook 信件）。
  ─ 注意：本頁含 SVG，「不可」直接貼進 Outlook 內文；Outlook 信件請沿用
    table+inline style 降級版（見 sales_brief.md 第三節），圖表區塊改為
    PNG <img> 或本頁連結。
  ─ 圖表數據為示意值（依 sales_brief.md 敘述性描述重建）；正式產出由
    MI3.0 / MI DataHub 取數程式填入，欄位對應見各 chart 區塊註解。
-->
<style>
*{box-sizing:border-box;margin:0;padding:0;}
body{font-family:'Segoe UI','Noto Sans TC','Microsoft JhengHei',Arial,sans-serif;background:#F3F4F6;color:#1A1A2E;font-size:14px;}
.container{max-width:960px;margin:0 auto;padding:16px;}
.header{background:linear-gradient(135deg,#1F4E79,#2E75B6);color:#fff;border-radius:12px;padding:26px 30px;margin-bottom:20px;}
.header h1{font-size:25px;font-weight:700;margin-bottom:8px;}
.header-meta{display:flex;flex-wrap:wrap;gap:12px;font-size:13px;opacity:0.92;}
.header-meta span{background:rgba(255,255,255,0.15);padding:4px 12px;border-radius:20px;}
.section-card{background:#fff;border-radius:12px;border:1px solid #C8D8E8;padding:22px 24px;margin-bottom:20px;}
.section-title{display:flex;align-items:center;gap:12px;margin-bottom:16px;}
.section-badge{width:28px;height:28px;border-radius:50%;background:#1F4E79;color:#fff;font-size:13px;font-weight:700;display:flex;align-items:center;justify-content:center;flex-shrink:0;}
.section-title h2{font-size:17px;font-weight:700;color:#1A1A2E;}
.sub-title{font-size:14px;font-weight:700;color:#1F4E79;margin:18px 0 10px;}
.chart-wrap{position:relative;padding-bottom:56%;height:0;overflow:hidden;border-radius:8px;border:1px solid #E0E0E0;background:#FCFCFB;}
.insight-box{background:#F5F7FA;border-left:4px solid #1F4E79;padding:12px 16px;border-radius:0 6px 6px 0;margin-top:14px;font-size:13px;line-height:1.8;color:#1A1A2E;}
table{width:100%;border-collapse:collapse;font-size:13px;}
th{background:#1F4E79;color:#fff;padding:8px 9px;text-align:center;font-weight:600;white-space:nowrap;}
td{padding:7px 9px;border-bottom:1px solid #DCE6F0;}
tr:nth-child(even) td{background:#EFF5FB;}
tr:nth-child(odd) td{background:#fff;}
.td-num{text-align:right;font-variant-numeric:tabular-nums;}
.td-txt{text-align:left;}
.tbl-scroll{overflow-x:auto;}
.tag-pos{display:inline-block;background:#E8F5E9;color:#2E7D32;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neg{display:inline-block;background:#FDECEC;color:#C00000;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neu{display:inline-block;background:#FFF8E1;color:#946C00;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.lamp{display:inline-block;width:11px;height:11px;border-radius:50%;vertical-align:middle;margin-right:6px;}
.note{font-size:11px;color:#8A8A96;margin-top:8px;line-height:1.6;}
.footer{text-align:center;font-size:12px;color:#9E9E9E;padding:14px 0 24px;line-height:1.7;}
.facet-label{width:84px;background:#EDF2F7 !important;color:#1F4E79;font-weight:700;vertical-align:top;}
</style>
</head>
<body>
<div class="container">

  <!-- ═══════════ Header ═══════════ -->
  <div class="header">
    <h1>Jul'26 Monitor Sales Brief</h1>
    <div class="header-meta">
      <span>&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測</span>
      <span>&#128202; 資料來源：MI DataHub / MI3.0</span>
      <span>&#128337; 數據截至：Jul'26 填報</span>
    </div>
  </div>

  <!-- 頂部提示 -->
  <div style="background:#FFF8E1;border:1px solid #FFD54F;border-radius:8px;padding:12px 18px;margin-bottom:18px;font-size:13px;color:#5D4037;line-height:1.7;">
    &#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。
  </div>

  <!-- ═══════════ 1｜品牌年度／季度 Panel buy 分析 ═══════════ -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">1</div><h2>品牌年度／季度 Panel buy 分析</h2></div>

    <div class="sub-title">1a｜年度 Panel Buy 總表（Unit: M pcs）</div>
    <div class="tbl-scroll">
      <table>
        <thead><tr>
          <th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th>
          <th>Y25 YoY%</th><th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th>
          <th>YTD (1~6M)</th><th>Hit%</th>
        </tr></thead>
        <tbody>
          <tr><td class="td-txt"><strong>Dell</strong></td><td class="td-num">23.5</td><td class="td-num">24.5</td><td class="td-num"><strong>24.5</strong></td><td class="td-num"><span class="tag-neg">-5%</span></td><td class="td-num">4%</td><td class="td-num"><span class="tag-pos">4%</span></td><td class="td-num">11.6</td><td class="td-num">47%</td></tr>
          <tr><td class="td-txt"><strong>HP</strong></td><td class="td-num">12.0</td><td class="td-num">11.0</td><td class="td-num"><strong>10.2</strong></td><td class="td-num"><span class="tag-neg">-23%</span></td><td class="td-num">-8%</td><td class="td-num"><span class="tag-neg">-15%</span></td><td class="td-num">6.4</td><td class="td-num">62%</td></tr>
          <tr><td class="td-txt"><strong>Samsung</strong></td><td class="td-num">11.7</td><td class="td-num">10.5</td><td class="td-num"><strong>10.5</strong></td><td class="td-num"><span class="tag-neg">-6%</span></td><td class="td-num">-10%</td><td class="td-num"><span class="tag-neg">-10%</span></td><td class="td-num">5.7</td><td class="td-num">54%</td></tr>
          <tr><td class="td-txt"><strong>Asus</strong></td><td class="td-num">6.3</td><td class="td-num">6.7</td><td class="td-num"><strong>6.7</strong></td><td class="td-num"><span class="tag-pos">27%</span></td><td class="td-num">6%</td><td class="td-num"><span class="tag-pos">6%</span></td><td class="td-num">3.7</td><td class="td-num">56%</td></tr>
        </tbody>
      </table>
    </div>
    <div class="insight-box">&#128204; <strong>AI 年度分析：</strong>Dell 為四品牌中唯一維持正成長者（Y26 24.5M、YoY +4%），且未再修正目標；YTD 11.6M 對應 Hit 47% 偏低，但其拉貨集中下半年，達成節奏尚在合理範圍。HP 目標連續下修（Pre -8% → Cur -15%），10.2M 較 Y25 再縮 1.8M，為四品牌最大減幅，Hit 62% 偏高反映的是「分母（年度目標）縮小」而非需求轉強。Samsung 維持 -10% 縮量，與其縮減中國 TV/monitor 業務方向一致。Asus 在 Y25 高成長（+27%）後轉為溫和擴張（+6%），YTD Hit 56% 進度正常。</div>

    <div class="sub-title">1b｜季度 Panel Buy 明細（Unit: M pcs）</div>
    <div class="tbl-scroll">
      <table>
        <thead><tr>
          <th>Brand</th><th>25 1Q</th><th>25 2Q</th><th>25 3Q</th><th>25 4Q</th>
          <th>26 1Q</th><th>26 2Q</th><th>26 3Q (Cur)</th>
          <th>26 1Q QoQ</th><th>26 2Q QoQ</th><th>26 3Q (Cur) QoQ</th>
        </tr></thead>
        <tbody>
          <tr><td class="td-txt"><strong>Dell</strong></td><td class="td-num">5.5</td><td class="td-num">6.0</td><td class="td-num">5.8</td><td class="td-num">6.1</td><td class="td-num">5.6</td><td class="td-num">6.0</td><td class="td-num"><strong>6.1</strong></td><td class="td-num"><span class="tag-neg">-8%</span></td><td class="td-num"><span class="tag-pos">+7%</span></td><td class="td-num"><span class="tag-pos">+2%</span></td></tr>
          <tr><td class="td-txt"><strong>HP</strong></td><td class="td-num">3.5</td><td class="td-num">3.0</td><td class="td-num">2.6</td><td class="td-num">3.0</td><td class="td-num">3.7</td><td class="td-num">2.7</td><td class="td-num"><strong>1.8</strong></td><td class="td-num"><span class="tag-pos">+23%</span></td><td class="td-num"><span class="tag-neg">-27%</span></td><td class="td-num"><span class="tag-neg">-33%</span></td></tr>
          <tr><td class="td-txt"><strong>Samsung</strong></td><td class="td-num">3.1</td><td class="td-num">2.8</td><td class="td-num">2.8</td><td class="td-num">3.0</td><td class="td-num">3.0</td><td class="td-num">2.7</td><td class="td-num"><strong>2.4</strong></td><td class="td-num"><span class="tag-neu">0%</span></td><td class="td-num"><span class="tag-neg">-10%</span></td><td class="td-num"><span class="tag-neg">-11%</span></td></tr>
          <tr><td class="td-txt"><strong>Asus</strong></td><td class="td-num">1.3</td><td class="td-num">1.7</td><td class="td-num">1.8</td><td class="td-num">1.5</td><td class="td-num">1.7</td><td class="td-num">2.0</td><td class="td-num"><strong>1.6</strong></td><td class="td-num"><span class="tag-pos">+13%</span></td><td class="td-num"><span class="tag-pos">+18%</span></td><td class="td-num"><span class="tag-neg">-20%</span></td></tr>
        </tbody>
      </table>
    </div>
    <div class="insight-box">&#128204; <strong>AI 季度分析：</strong>Dell 26-1Q 季節性回落（QoQ -8%）後連兩季回升（+7%、+2%），26-3Q 6.1M 重回 25-4Q 高點，高檔動能延續。HP 走勢反差最大：26-1Q +23% 衝高（IC 缺料提前拉貨）後急轉直下，連兩季 -27%、-33%，26-3Q 1.8M 為近三年單季低點，與年度目標下修方向一致，需留意前期拉貨轉庫存的消化壓力。Samsung 連兩季 -10%／-11% 緩步收縮，與縮減中國業務的年度基調吻合。Asus 26-2Q +18% 拉至 2.0M 高點後 26-3Q -20% 回落至 1.6M，屬旺季後正常修正。</div>
    <div class="note">註：QoQ 為對前一季變化率（26 1Q QoQ 以 25 4Q 為基期）；(Pre)＝上次填報、(Cur)＝本次填報；Hit%＝YTD 對 Y26 (Cur) 年度目標達成率。</div>
  </div>

  <!-- ═══════════ 2｜品牌月度面板採購 ═══════════ -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">2</div><h2>品牌月度面板採購</h2></div>

    <div class="sub-title">2a｜MI3.0 月度採購趨勢（25-Jan ～ 26-Sep(F)，Unit: M pcs）</div>
    <!-- CHART1: 由 build_charts.py 依 MI3.0 取數結果重生；目前為示意數據 -->
    <!--CHART1:BEGIN-->
<div class="chart-wrap"><svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 720 400" role="img" aria-label="四品牌月度面板採購趨勢（MI3.0）" style="position:absolute;top:0;left:0;display:block;">
<text x="337.0" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#1A1A2E">四品牌月度面板採購趨勢（MI3.0）</text>
<text x="46" y="24" text-anchor="start" font-size="11" fill="#898781">Unit: M pcs</text>
<rect x="540.7" y="44" width="93.3" height="288" fill="#F0F3F7"/>
<text x="584.4" y="56" text-anchor="middle" font-size="10" fill="#898781">Forecast</text>
<line x1="46" y1="332.0" x2="628" y2="332.0" stroke="#C3C2B7" stroke-width="1"/>
<text x="40" y="336.0" text-anchor="end" font-size="11" fill="#898781">0</text>
<line x1="46" y1="249.7" x2="628" y2="249.7" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="253.7" text-anchor="end" font-size="11" fill="#898781">1</text>
<line x1="46" y1="167.4" x2="628" y2="167.4" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="171.4" text-anchor="end" font-size="11" fill="#898781">2</text>
<line x1="46" y1="85.1" x2="628" y2="85.1" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="89.1" text-anchor="end" font-size="11" fill="#898781">3</text>
<text x="46.0" y="350" text-anchor="middle" font-size="11" fill="#898781">25-Jan</text>
<text x="133.3" y="350" text-anchor="middle" font-size="11" fill="#898781">Apr</text>
<text x="220.6" y="350" text-anchor="middle" font-size="11" fill="#898781">Jul</text>
<text x="307.9" y="350" text-anchor="middle" font-size="11" fill="#898781">Oct</text>
<text x="395.2" y="350" text-anchor="middle" font-size="11" fill="#898781">26-Jan</text>
<text x="482.5" y="350" text-anchor="middle" font-size="11" fill="#898781">Apr</text>
<text x="569.8" y="350" text-anchor="middle" font-size="11" fill="#898781">Jul(F)</text>
<text x="628.0" y="350" text-anchor="middle" font-size="11" fill="#898781">Sep(F)</text>
<polyline points="46.0,134.5 75.1,208.6 104.2,175.7 133.3,101.6 162.4,225.0 191.5,249.7 220.6,52.2 249.7,216.8 278.8,290.9 307.9,76.9 337.0,118.1 366.1,76.9 395.2,249.7 424.3,233.3 453.4,208.6 482.5,216.8 511.6,85.1 540.7,101.6" fill="none" stroke="#2a78d6" stroke-width="2" stroke-linejoin="round"/>
<polyline points="540.7,101.6 569.8,118.1 598.9,93.4 628.0,126.3" fill="none" stroke="#2a78d6" stroke-width="2" stroke-dasharray="5 4" stroke-linejoin="round"/>
<circle cx="628.0" cy="126.3" r="3.5" fill="#2a78d6" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="130.3" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">Dell 2.5</text>
<polyline points="46.0,266.2 75.1,262.1 104.2,257.9 133.3,266.2 162.4,282.6 191.5,290.9 220.6,295.0 249.7,290.9 278.8,286.7 307.9,282.6 337.0,290.9 366.1,290.9 395.2,299.1 424.3,286.7 453.4,295.0 482.5,299.1 511.6,295.0 540.7,303.2" fill="none" stroke="#eb6834" stroke-width="2" stroke-linejoin="round"/>
<polyline points="540.7,303.2 569.8,303.2 598.9,304.8 628.0,307.3" fill="none" stroke="#eb6834" stroke-width="2" stroke-dasharray="5 4" stroke-linejoin="round"/>
<circle cx="628.0" cy="307.3" r="3.5" fill="#eb6834" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="311.3" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">Samsung 0.3</text>
<polyline points="46.0,241.5 75.1,241.5 104.2,233.3 133.3,225.0 162.4,233.3 191.5,249.7 220.6,266.2 249.7,225.0 278.8,233.3 307.9,241.5 337.0,208.6 366.1,225.0 395.2,249.7 424.3,266.2 453.4,249.7 482.5,257.9 511.6,274.4 540.7,266.2" fill="none" stroke="#1baf7a" stroke-width="2" stroke-linejoin="round"/>
<polyline points="540.7,266.2 569.8,266.2 598.9,257.9 628.0,257.9" fill="none" stroke="#1baf7a" stroke-width="2" stroke-dasharray="5 4" stroke-linejoin="round"/>
<circle cx="628.0" cy="257.9" r="3.5" fill="#1baf7a" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="261.9" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">HP 0.9</text>
<polyline points="46.0,282.6 75.1,278.5 104.2,266.2 133.3,253.8 162.4,249.7 191.5,249.7 220.6,262.1 249.7,274.4 278.8,278.5 307.9,262.1 337.0,257.9 366.1,274.4 395.2,257.9 424.3,262.1 453.4,249.7 482.5,249.7 511.6,262.1 540.7,274.4" fill="none" stroke="#eda100" stroke-width="2" stroke-linejoin="round"/>
<polyline points="540.7,274.4 569.8,274.4 598.9,274.4 628.0,274.4" fill="none" stroke="#eda100" stroke-width="2" stroke-dasharray="5 4" stroke-linejoin="round"/>
<circle cx="628.0" cy="274.4" r="3.5" fill="#eda100" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="278.4" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">ASUS 0.7</text>
<line x1="86" y1="386" x2="104" y2="386" stroke="#2a78d6" stroke-width="3"/>
<text x="110" y="390" font-size="11" fill="#52514E">Dell</text>
<line x1="172" y1="386" x2="190" y2="386" stroke="#eb6834" stroke-width="3"/>
<text x="196" y="390" font-size="11" fill="#52514E">Samsung</text>
<line x1="280" y1="386" x2="298" y2="386" stroke="#1baf7a" stroke-width="3"/>
<text x="304" y="390" font-size="11" fill="#52514E">HP</text>
<line x1="355" y1="386" x2="373" y2="386" stroke="#eda100" stroke-width="3"/>
<text x="379" y="390" font-size="11" fill="#52514E">ASUS</text>
</svg></div>
<!--CHART1:END-->
    <div class="insight-box">&#128204; <strong>趨勢判讀：</strong>Dell 呈明顯季節性脈衝式拉貨（峰值 3.4M、谷底 0.5M），2026 年 5 月起再度進入拉貨波段；HP 2026 年起走弱，近期落在 0.8M 附近；Samsung 低位續探 0.3–0.5M 區間；ASUS 於 0.5–1.0M 間小幅波動、走勢平穩。</div>

    <div class="sub-title">2b｜MoM% 變化表（近 9 個月，A=Actual、F=Forecast）</div>
    <div class="tbl-scroll">
      <table>
        <thead><tr>
          <th>MoM</th><th>26-Jan</th><th>Feb</th><th>Mar</th><th>Apr</th><th>May</th><th>Jun(A)</th><th>Jul(F)</th><th>Aug(F)</th><th>Sep(F)</th>
        </tr></thead>
        <tbody>
          <tr><td class="td-txt"><strong>Major 4</strong></td><td class="td-num">10%</td><td class="td-num">15%</td><td class="td-num"><span class="tag-neg">-18%</span></td><td class="td-num">26%</td><td class="td-num"><span class="tag-neg">-26%</span></td><td class="td-num"><span class="tag-neg">-25%</span></td><td class="td-num"><span class="tag-pos">46%</span></td><td class="td-num">-7%</td><td class="td-num">-9%</td></tr>
          <tr><td class="td-txt"><strong>Dell</strong></td><td class="td-num"><span class="tag-neg">-69%</span></td><td class="td-num">20%</td><td class="td-num">24%</td><td class="td-num">-8%</td><td class="td-num"><span class="tag-pos">53%</span></td><td class="td-num">-7%</td><td class="td-num"><span class="tag-pos">77%</span></td><td class="td-num"><span class="tag-pos">178%</span></td><td class="td-num">-13%</td></tr>
          <tr><td class="td-txt"><strong>HP</strong></td><td class="td-num"><span class="tag-neg">-23%</span></td><td class="td-num"><span class="tag-neg">-26%</span></td><td class="td-num">34%</td><td class="td-num">-13%</td><td class="td-num"><span class="tag-neg">-23%</span></td><td class="td-num">13%</td><td class="td-num"><span class="tag-neg">-69%</span></td><td class="td-num"><span class="tag-pos">178%</span></td><td class="td-num">4%</td></tr>
          <tr><td class="td-txt"><strong>Samsung</strong></td><td class="td-num"><span class="tag-neg">-18%</span></td><td class="td-num">33%</td><td class="td-num">-17%</td><td class="td-num">-10%</td><td class="td-num">11%</td><td class="td-num"><span class="tag-neg">-20%</span></td><td class="td-num">0%</td><td class="td-num">-6%</td><td class="td-num">-9%</td></tr>
          <tr><td class="td-txt"><strong>ASUS</strong></td><td class="td-num">30%</td><td class="td-num">-8%</td><td class="td-num">17%</td><td class="td-num">0%</td><td class="td-num">-14%</td><td class="td-num"><span class="tag-neg">-17%</span></td><td class="td-num">0%</td><td class="td-num">0%</td><td class="td-num">0%</td></tr>
        </tbody>
      </table>
    </div>
    <div class="note">註：前版 Aug／Sep 出現空格為報表「取數區間截斷」問題（MI3.0 原始數據正常）；本版取數規則改為「以 MI3.0 該品牌實際有值月份為準」，僅當 MI3.0 亦無資料時顯示「-」。表列灰字斜體值為待 MI3.0 回補之示意值。</div>
  </div>

  <!-- ═══════════ 3｜業務訊息整體摘要 ═══════════ -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">3</div><h2>業務訊息整體摘要</h2></div>
    <table>
      <tbody>
        <tr><td class="facet-label">整體市況</td><td class="td-txt">ASUS 北美 Q1 底、歐洲 Q2、中國 Q3 需求依序下滑；Dell 預估 2026 全球 monitor 出貨衰退 1–2%；Samsung 持續縮減中國 TV/monitor 業務，中國品牌競爭壓力限制其出貨回補。</td></tr>
        <tr><td class="facet-label">產品技術</td><td class="td-txt">HP 商用產品全面升級 100Hz→120Hz，預計 2026/E 量產；ASUS 期望友達 M270QAN06.0 推出 4ch LVDS 版本做衝量機種；Dell 下半年擴大消費性產品佈局。</td></tr>
        <tr><td class="facet-label">價格相關</td><td class="td-txt">MSI 在中國 BP 翻倍目標致售價貼近白牌；27" QHD 240Hz 以上 OLED 競廠價格積極；Dell 因 IC 缺料拉貨面板但同時因高庫存壓供應商降價；HP 以競標最低價者得 70% 占比。</td></tr>
        <tr><td class="facet-label">其他焦點</td><td class="td-txt">Dell 正式啟動越南 PCB/PCBA 產線供 DAO 區域，目標 2027 年 1 月整機出貨；HP 因記憶體短缺 AIO 需內部分配，MNT 產品線維持要求面板供應商 12 WOS IC 策略備料方向。</td></tr>
      </tbody>
    </table>
  </div>

  <!-- ═══════════ 4｜各品牌市場訊息矩陣 ═══════════ -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">4</div><h2>各品牌市場訊息矩陣</h2></div>
    <div class="tbl-scroll">
      <table>
        <thead><tr><th style="width:64px;">品牌</th><th style="width:26%;">市場展望</th><th style="width:26%;">產品技術</th><th style="width:23%;">價格</th><th>其他資訊</th></tr></thead>
        <tbody>
          <tr style="vertical-align:top;">
            <td class="td-txt"><strong>ASUS</strong></td>
            <td class="td-txt">北美 Q1 底、歐洲 Q2、中國 618 後 Q3 需求依序往下，全球逐季衰退。</td>
            <td class="td-txt">M270QAN06.0 希望友達提供 4ch LVDS 版本，放入 VG 系列作衝量戰鬥機種。</td>
            <td class="td-txt">MSI 中國 BP 目標翻倍，售價貼近白牌；27" QHD 240Hz+ OLED 競廠價格積極。</td>
            <td class="td-txt" style="color:#9E9E9E;">（無填寫）</td>
          </tr>
          <tr style="vertical-align:top;">
            <td class="td-txt"><strong>Dell</strong></td>
            <td class="td-txt">2026 全球顯示器出貨預估衰退 1–2%，下半年因缺料及 PC 漲價需求前景不確定。</td>
            <td class="td-txt">目標 2H 2026 推出更多消費性產品，提升消費性市佔。</td>
            <td class="td-txt">IC 缺料致採購拉貨提前，但高庫存迫使供應商降價。</td>
            <td class="td-txt">越南 PCB/PCBA 正式啟動供 DAO 區域，目標 2027/1 整機出貨。</td>
          </tr>
          <tr style="vertical-align:top;">
            <td class="td-txt"><strong>HP</strong></td>
            <td class="td-txt">AI PC 與 Win 11 換機潮為成長動能，寄望 bundle 模式帶動顯示器需求。</td>
            <td class="td-txt">主力商用產品刷新率 100Hz→120Hz，預計 2026/E 量產。</td>
            <td class="td-txt">RFQ 持續議價，已量產機種以競標最低價者取得 70% 占比降本。</td>
            <td class="td-txt">AIO 因記憶體缺料需內部分配；MNT 維持 12 週 IC 策略備貨。</td>
          </tr>
          <tr style="vertical-align:top;">
            <td class="td-txt"><strong>Samsung</strong></td>
            <td class="td-txt">預計進一步縮減中國 TV/monitor 業務，中國市佔受中國品牌競爭壓力限制復甦。</td>
            <td class="td-txt">高階市佔仍強，但整體出貨量市佔持續受挑戰。</td>
            <td class="td-txt" style="color:#9E9E9E;">（無填寫）</td>
            <td class="td-txt" style="color:#9E9E9E;">（無填寫）</td>
          </tr>
        </tbody>
      </table>
    </div>
  </div>

  <!-- ═══════════ 5｜主要品牌庫存週數 ═══════════ -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">5</div><h2>主要品牌庫存週數（Jul'26 pipeline）</h2></div>

    <div class="sub-title">5a｜MI3.0 庫存週數趨勢（2025-07 ～ 2026-07，Unit: week）</div>
    <!-- CHART2: 由 build_charts.py 依 MI3.0 取數結果重生；目前為示意數據 -->
    <!--CHART2:BEGIN-->
<div class="chart-wrap"><svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 720 400" role="img" aria-label="主要品牌庫存週數趨勢（MI3.0）" style="position:absolute;top:0;left:0;display:block;">
<text x="337.0" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#1A1A2E">主要品牌庫存週數趨勢（MI3.0）</text>
<text x="46" y="24" text-anchor="start" font-size="11" fill="#898781">Unit: week</text>
<line x1="46" y1="332.0" x2="628" y2="332.0" stroke="#C3C2B7" stroke-width="1"/>
<text x="40" y="336.0" text-anchor="end" font-size="11" fill="#898781">0</text>
<line x1="46" y1="276.6" x2="628" y2="276.6" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="280.6" text-anchor="end" font-size="11" fill="#898781">5</text>
<line x1="46" y1="221.2" x2="628" y2="221.2" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="225.2" text-anchor="end" font-size="11" fill="#898781">10</text>
<line x1="46" y1="165.8" x2="628" y2="165.8" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="169.8" text-anchor="end" font-size="11" fill="#898781">15</text>
<line x1="46" y1="110.5" x2="628" y2="110.5" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="114.5" text-anchor="end" font-size="11" fill="#898781">20</text>
<line x1="46" y1="55.1" x2="628" y2="55.1" stroke="#E1E0D9" stroke-dasharray="3 3" stroke-width="1"/>
<text x="40" y="59.1" text-anchor="end" font-size="11" fill="#898781">25</text>
<text x="46.0" y="350" text-anchor="middle" font-size="11" fill="#898781">25-07</text>
<text x="143.0" y="350" text-anchor="middle" font-size="11" fill="#898781">25-09</text>
<text x="240.0" y="350" text-anchor="middle" font-size="11" fill="#898781">25-11</text>
<text x="337.0" y="350" text-anchor="middle" font-size="11" fill="#898781">26-01</text>
<text x="434.0" y="350" text-anchor="middle" font-size="11" fill="#898781">26-03</text>
<text x="531.0" y="350" text-anchor="middle" font-size="11" fill="#898781">26-05</text>
<text x="628.0" y="350" text-anchor="middle" font-size="11" fill="#898781">26-07</text>
<polyline points="46.0,55.1 94.5,55.1 143.0,60.6 191.5,60.6 240.0,66.2 288.5,66.2 337.0,66.2 385.5,66.2 434.0,66.2 482.5,66.2 531.0,66.2 579.5,60.6 628.0,55.1" fill="none" stroke="#2a78d6" stroke-width="2" stroke-linejoin="round"/>
<circle cx="628.0" cy="55.1" r="3.5" fill="#2a78d6" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="59.1" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">Dell 25</text>
<polyline points="46.0,188.0 94.5,193.5 143.0,199.1 191.5,204.6 240.0,210.2 288.5,215.7 337.0,221.2 385.5,221.2 434.0,215.7 482.5,210.2 531.0,199.1 579.5,182.5 628.0,165.8" fill="none" stroke="#eb6834" stroke-width="2" stroke-linejoin="round"/>
<circle cx="628.0" cy="165.8" r="3.5" fill="#eb6834" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="169.8" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">Samsung 15</text>
<polyline points="46.0,199.1 94.5,204.6 143.0,210.2 191.5,215.7 240.0,221.2 288.5,226.8 337.0,237.8 385.5,232.3 434.0,221.2 482.5,210.2 531.0,199.1 579.5,188.0 628.0,176.9" fill="none" stroke="#1baf7a" stroke-width="2" stroke-linejoin="round"/>
<circle cx="628.0" cy="176.9" r="3.5" fill="#1baf7a" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="183.8" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">HP 14</text>
<polyline points="46.0,104.9 94.5,99.4 143.0,104.9 191.5,110.5 240.0,121.5 288.5,132.6 337.0,132.6 385.5,132.6 434.0,127.1 482.5,132.6 531.0,116.0 579.5,88.3 628.0,66.2" fill="none" stroke="#eda100" stroke-width="2" stroke-linejoin="round"/>
<circle cx="628.0" cy="66.2" r="3.5" fill="#eda100" stroke="#FCFCFB" stroke-width="2"/>
<text x="636.0" y="73.1" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">ASUS 24</text>
<line x1="86" y1="386" x2="104" y2="386" stroke="#2a78d6" stroke-width="3"/>
<text x="110" y="390" font-size="11" fill="#52514E">Dell</text>
<line x1="172" y1="386" x2="190" y2="386" stroke="#eb6834" stroke-width="3"/>
<text x="196" y="390" font-size="11" fill="#52514E">Samsung</text>
<line x1="280" y1="386" x2="298" y2="386" stroke="#1baf7a" stroke-width="3"/>
<text x="304" y="390" font-size="11" fill="#52514E">HP</text>
<line x1="355" y1="386" x2="373" y2="386" stroke="#eda100" stroke-width="3"/>
<text x="379" y="390" font-size="11" fill="#52514E">ASUS</text>
</svg></div>
<!--CHART2:END-->
    <div class="insight-box">&#128204; <strong>趨勢判讀：</strong>Dell 全期 24–25w 高位盤整未見去化；ASUS 自 2026-04 起由 18w 急升至 24w，與 Dell 收斂並進入紅燈區；HP 於 2026-01 觸底 8.5w 後強力反彈至 14w；Samsung 自 10w 回升至 15w，兩者升勢皆入黃燈觀察區。</div>

    <div class="sub-title">5b｜Jul'26 庫存燈號</div>
    <table>
      <thead><tr><th style="width:64px;">燈號</th><th style="width:110px;">品牌</th><th style="width:90px;">週數</th><th class="td-txt">判讀</th></tr></thead>
      <tbody>
        <tr><td style="text-align:center;"><span class="lamp" style="background:#C00000;"></span><strong style="color:#C00000;">紅燈</strong></td><td class="td-txt"><strong>Dell</strong></td><td class="td-num"><strong>25w</strong></td><td class="td-txt">庫存偏高，高位盤整未去化</td></tr>
        <tr><td style="text-align:center;"><span class="lamp" style="background:#C00000;"></span><strong style="color:#C00000;">紅燈</strong></td><td class="td-txt"><strong>ASUS</strong></td><td class="td-num"><strong>24w</strong></td><td class="td-txt">庫存偏高，近月急升與 Dell 收斂</td></tr>
        <tr><td style="text-align:center;"><span class="lamp" style="background:#BF8F00;"></span><strong style="color:#946C00;">黃燈</strong></td><td class="td-txt"><strong>Samsung</strong></td><td class="td-num"><strong>15w</strong></td><td class="td-txt">庫存升高，需謹慎觀察</td></tr>
        <tr><td style="text-align:center;"><span class="lamp" style="background:#BF8F00;"></span><strong style="color:#946C00;">黃燈</strong></td><td class="td-txt"><strong>HP</strong></td><td class="td-num"><strong>14w</strong></td><td class="td-txt">庫存升高（自 8.5w 低點反彈），需謹慎觀察</td></tr>
      </tbody>
    </table>
  </div>

  <!-- ═══════════ Footer ═══════════ -->
  <div class="footer">
    本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 Jul'26 填報。<br>
    Outlook 信件版為表格降級版；本頁為完整圖表版（信件內「查看完整圖表」連結之落地頁）。互動篩選請至 Databricks Dashboard。
  </div>

</div>
</body>
</html>
'''

In [ ]:
MAIL_BODY_TEMPLATE = '''
<!DOCTYPE html>
<html xmlns:o="urn:schemas-microsoft-com:office:office" lang=zh-Hant><head><meta charset=UTF-8><title>Jul'26 Monitor Sales Brief</title>
<!--
Sales Brief Email 版模板（Outlook 相容 table 降級版，與 ATTACHMENT_TEMPLATE 同源同資訊）
─ v2＝瀏覽器完整版（SVG 互動圖，隨信 .html 附件）；本檔＝send_mail 內文，寬 640px。
─ 圖表為 PNG 佔位符 {{CHART1_PNG_URL}} / {{CHART2_PNG_URL}}（由 v2 截圖或 generate_chart 產出 https URL 後替換）。
─ 數據為示意值（與 v2 一致），正式產出由 MI3.0 / MI DataHub 取數程式填入。
─ 樣式集中於 head <style>（Outlook 桌面 / OWA / Gmail 均支援）以最小化字元數；剝除 style 的客戶端退化為素表但內容完整。
-->
<!--[if mso]><noscript><xml><o:OfficeDocumentSettings><o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml></noscript><![endif]-->
<style>
body{margin:0;background:#F3F4F6}
table{border-collapse:collapse}
td,th{font-family:'Microsoft JhengHei','Noto Sans TC',sans-serif;font-size:12px}
th{background:#1F4E79;color:#FFFFFF;padding:5px 3px;font-size:11px}
td{padding:5px 4px;border-bottom:1px solid #DCE6F0;text-align:right}
.t{text-align:left}
.ce{text-align:center}
.w{padding:18px 22px;border:0;text-align:left}
.s{padding:0;border:0;font-size:0;line-height:0}
.b{background:#1F4E79;color:#FFFFFF;font-weight:700;text-align:center;border-radius:50%;padding:0;border:0}
.h{font-size:16px;font-weight:700;padding:0 0 0 10px;border:0;text-align:left}
.st{color:#1F4E79;font-weight:700;font-size:13px;padding:14px 0 7px;border:0;text-align:left}
.i{background:#F5F7FA;padding:10px 14px;line-height:1.8;border:0;text-align:left}
.no{font-size:11px;color:#8A8A96;padding:6px 0 0;border:0;text-align:left;line-height:1.6}
.fl{background:#EDF2F7;color:#1F4E79;font-weight:700;text-align:left;vertical-align:top;width:76px}
.p{background:#E8F5E9;color:#2E7D32;border-radius:8px;padding:0 5px;font-size:11px}
.n{background:#FDECEC;color:#C00000;border-radius:8px;padding:0 5px;font-size:11px}
.u{background:#FFF8E1;color:#946C00;border-radius:8px;padding:0 5px;font-size:11px}
.f{font-size:11px;color:#9E9E9E;text-align:center;line-height:1.7;padding:12px 0 20px;border:0}
</style></head>
<body bgcolor=#F3F4F6>
<table width=100% cellpadding=0 cellspacing=0 bgcolor=#F3F4F6><tr><td class=s align=center>
<table width=640 cellpadding=0 cellspacing=0>
<tr><td class=s height=16></td></tr>
<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#1F4E79><tr><td class=w style="padding:22px 24px">
<span style="font-size:22px;font-weight:700;color:#FFFFFF">Jul'26 Monitor Sales Brief</span><br>
<span style="font-size:12px;color:#CFE0F0;line-height:2">&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測　&#128202; 資料來源：MI DataHub / MI3.0　&#128337; 數據截至：Jul'26 填報</span>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>
<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFF8E1 style="border:1px solid #FFD54F"><tr><td class=w style="padding:10px 16px;color:#5D4037;line-height:1.7">&#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>

<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>1</td><td class=h>品牌年度／季度 Panel buy 分析</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1a｜年度 Panel Buy 總表（Unit: M pcs）</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th><th>Y25 YoY%</th><th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th><th>YTD (1~6M)</th><th>Hit%</th></tr>
<tr><td class=t><b>Dell</b></td><td>23.5</td><td>24.5</td><td><b>24.5</b></td><td><span class=n>-5%</span></td><td>4%</td><td><span class=p>4%</span></td><td>11.6</td><td>47%</td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>HP</b></td><td>12.0</td><td>11.0</td><td><b>10.2</b></td><td><span class=n>-23%</span></td><td>-8%</td><td><span class=n>-15%</span></td><td>6.4</td><td>62%</td></tr>
<tr><td class=t><b>Samsung</b></td><td>11.7</td><td>10.5</td><td><b>10.5</b></td><td><span class=n>-6%</span></td><td>-10%</td><td><span class=n>-10%</span></td><td>5.7</td><td>54%</td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>Asus</b></td><td>6.3</td><td>6.7</td><td><b>6.7</b></td><td><span class=p>27%</span></td><td>6%</td><td><span class=p>6%</span></td><td>3.7</td><td>56%</td></tr>
</table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 年度分析：</b>Dell 為四品牌中唯一維持正成長者（Y26 24.5M、YoY +4%），且未再修正目標；YTD 11.6M 對應 Hit 47% 偏低，但其拉貨集中下半年，達成節奏尚在合理範圍。HP 目標連續下修（Pre -8% → Cur -15%），10.2M 較 Y25 再縮 1.8M，為四品牌最大減幅，Hit 62% 偏高反映的是「分母（年度目標）縮小」而非需求轉強。Samsung 維持 -10% 縮量，與其縮減中國 TV/monitor 業務方向一致。Asus 在 Y25 高成長（+27%）後轉為溫和擴張（+6%），YTD Hit 56% 進度正常。</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1b｜季度 Panel Buy 明細（Unit: M pcs）</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><th>Brand</th><th>25 1Q</th><th>25 2Q</th><th>25 3Q</th><th>25 4Q</th><th>26 1Q</th><th>26 2Q</th><th>26 3Q (Cur)</th><th>26 1Q QoQ</th><th>26 2Q QoQ</th><th>26 3Q (Cur) QoQ</th></tr>
<tr><td class=t><b>Dell</b></td><td>5.5</td><td>6.0</td><td>5.8</td><td>6.1</td><td>5.6</td><td>6.0</td><td><b>6.1</b></td><td><span class=n>-8%</span></td><td><span class=p>+7%</span></td><td><span class=p>+2%</span></td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>HP</b></td><td>3.5</td><td>3.0</td><td>2.6</td><td>3.0</td><td>3.7</td><td>2.7</td><td><b>1.8</b></td><td><span class=p>+23%</span></td><td><span class=n>-27%</span></td><td><span class=n>-33%</span></td></tr>
<tr><td class=t><b>Samsung</b></td><td>3.1</td><td>2.8</td><td>2.8</td><td>3.0</td><td>3.0</td><td>2.7</td><td><b>2.4</b></td><td><span class=u>0%</span></td><td><span class=n>-10%</span></td><td><span class=n>-11%</span></td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>Asus</b></td><td>1.3</td><td>1.7</td><td>1.8</td><td>1.5</td><td>1.7</td><td>2.0</td><td><b>1.6</b></td><td><span class=p>+13%</span></td><td><span class=p>+18%</span></td><td><span class=n>-20%</span></td></tr>
</table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 季度分析：</b>Dell 26-1Q 季節性回落（QoQ -8%）後連兩季回升（+7%、+2%），26-3Q 6.1M 重回 25-4Q 高點，高檔動能延續。HP 走勢反差最大：26-1Q +23% 衝高（IC 缺料提前拉貨）後急轉直下，連兩季 -27%、-33%，26-3Q 1.8M 為近三年單季低點，與年度目標下修方向一致，需留意前期拉貨轉庫存的消化壓力。Samsung 連兩季 -10%／-11% 緩步收縮，與縮減中國業務的年度基調吻合。Asus 26-2Q +18% 拉至 2.0M 高點後 26-3Q -20% 回落至 1.6M，屬旺季後正常修正。</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：QoQ 為對前一季變化率（26 1Q QoQ 以 25 4Q 為基期）；(Pre)＝上次填報、(Cur)＝本次填報；Hit%＝YTD 對 Y26 (Cur) 年度目標達成率。</td></tr></table>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>

<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>2</td><td class=h>品牌月度面板採購</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2a｜MI3.0 月度採購趨勢（25-Jan ～ 26-Sep(F)，Unit: M pcs）</td></tr></table>
<!--CHART1: PNG 由 v2 截圖或 generate_chart 產出後替換 src-->
<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART1_PNG_URL}}" width=590 height=328 alt="四品牌月度面板採購趨勢（MI3.0），Unit: M pcs" style="display:block;border:0;max-width:100%"></td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>趨勢判讀：</b>Dell 呈明顯季節性脈衝式拉貨（峰值 3.4M、谷底 0.5M），2026 年 5 月起再度進入拉貨波段；HP 2026 年起走弱，近期落在 0.8M 附近；Samsung 低位續探 0.3–0.5M 區間；ASUS 於 0.5–1.0M 間小幅波動、走勢平穩。</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2b｜MoM% 變化表（近 9 個月，A=Actual、F=Forecast）</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><th>MoM</th><th>26-Jan</th><th>Feb</th><th>Mar</th><th>Apr</th><th>May</th><th>Jun(A)</th><th>Jul(F)</th><th>Aug(F)</th><th>Sep(F)</th></tr>
<tr><td class=t><b>Major 4</b></td><td>10%</td><td>15%</td><td><span class=n>-18%</span></td><td>26%</td><td><span class=n>-26%</span></td><td><span class=n>-25%</span></td><td><span class=p>46%</span></td><td>-7%</td><td>-9%</td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>Dell</b></td><td><span class=n>-69%</span></td><td>20%</td><td>24%</td><td>-8%</td><td><span class=p>53%</span></td><td>-7%</td><td><span class=p>77%</span></td><td><span class=p>178%</span></td><td>-13%</td></tr>
<tr><td class=t><b>HP</b></td><td><span class=n>-23%</span></td><td><span class=n>-26%</span></td><td>34%</td><td>-13%</td><td><span class=n>-23%</span></td><td>13%</td><td><span class=n>-69%</span></td><td><span class=p>178%</span></td><td>4%</td></tr>
<tr bgcolor=#EFF5FB><td class=t><b>Samsung</b></td><td><span class=n>-18%</span></td><td>33%</td><td>-17%</td><td>-10%</td><td>11%</td><td><span class=n>-20%</span></td><td>0%</td><td>-6%</td><td>-9%</td></tr>
<tr><td class=t><b>ASUS</b></td><td>30%</td><td>-8%</td><td>17%</td><td>0%</td><td>-14%</td><td><span class=n>-17%</span></td><td>0%</td><td>0%</td><td>0%</td></tr>
</table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：前版 Aug／Sep 出現空格為報表「取數區間截斷」問題（MI3.0 原始數據正常）；本版取數規則改為「以 MI3.0 該品牌實際有值月份為準」，僅當 MI3.0 亦無資料時顯示「-」。表列灰字斜體值為待 MI3.0 回補之示意值。</td></tr></table>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>

<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>3</td><td class=h>業務訊息整體摘要</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><td class=s height=10 colspan=2></td></tr>
<tr><td class=fl>整體市況</td><td class=t style="vertical-align:top">ASUS 北美 Q1 底、歐洲 Q2、中國 Q3 需求依序下滑；Dell 預估 2026 全球 monitor 出貨衰退 1–2%；Samsung 持續縮減中國 TV/monitor 業務，中國品牌競爭壓力限制其出貨回補。</td></tr>
<tr bgcolor=#EFF5FB><td class=fl>產品技術</td><td class=t style="vertical-align:top">HP 商用產品全面升級 100Hz→120Hz，預計 2026/E 量產；ASUS 期望友達 M270QAN06.0 推出 4ch LVDS 版本做衝量機種；Dell 下半年擴大消費性產品佈局。</td></tr>
<tr><td class=fl>價格相關</td><td class=t style="vertical-align:top">MSI 在中國 BP 翻倍目標致售價貼近白牌；27" QHD 240Hz 以上 OLED 競廠價格積極；Dell 因 IC 缺料拉貨面板但同時因高庫存壓供應商降價；HP 以競標最低價者得 70% 占比。</td></tr>
<tr bgcolor=#EFF5FB><td class=fl>其他焦點</td><td class=t style="vertical-align:top">Dell 正式啟動越南 PCB/PCBA 產線供 DAO 區域，目標 2027 年 1 月整機出貨；HP 因記憶體短缺 AIO 需內部分配，MNT 產品線維持要求面板供應商 12 WOS IC 策略備料方向。</td></tr>
</table>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>

<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>4</td><td class=h>各品牌市場訊息矩陣</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><td class=s height=10 colspan=5></td></tr>
<tr><th width=44>品牌</th><th width=145>市場展望</th><th width=145>產品技術</th><th width=130>價格</th><th>其他資訊</th></tr>
<tr valign=top><td class=t><b>ASUS</b></td><td class=t>北美 Q1 底、歐洲 Q2、中國 618 後 Q3 需求依序往下，全球逐季衰退。</td><td class=t>M270QAN06.0 希望友達提供 4ch LVDS 版本，放入 VG 系列作衝量戰鬥機種。</td><td class=t>MSI 中國 BP 目標翻倍，售價貼近白牌；27" QHD 240Hz+ OLED 競廠價格積極。</td><td class=t style="color:#9E9E9E">（無填寫）</td></tr>
<tr valign=top bgcolor=#EFF5FB><td class=t><b>Dell</b></td><td class=t>2026 全球顯示器出貨預估衰退 1–2%，下半年因缺料及 PC 漲價需求前景不確定。</td><td class=t>目標 2H 2026 推出更多消費性產品，提升消費性市佔。</td><td class=t>IC 缺料致採購拉貨提前，但高庫存迫使供應商降價。</td><td class=t>越南 PCB/PCBA 正式啟動供 DAO 區域，目標 2027/1 整機出貨。</td></tr>
<tr valign=top><td class=t><b>HP</b></td><td class=t>AI PC 與 Win 11 換機潮為成長動能，寄望 bundle 模式帶動顯示器需求。</td><td class=t>主力商用產品刷新率 100Hz→120Hz，預計 2026/E 量產。</td><td class=t>RFQ 持續議價，已量產機種以競標最低價者取得 70% 占比降本。</td><td class=t>AIO 因記憶體缺料需內部分配；MNT 維持 12 週 IC 策略備貨。</td></tr>
<tr valign=top bgcolor=#EFF5FB><td class=t><b>Samsung</b></td><td class=t>預計進一步縮減中國 TV/monitor 業務，中國市佔受中國品牌競爭壓力限制復甦。</td><td class=t>高階市佔仍強，但整體出貨量市佔持續受挑戰。</td><td class=t style="color:#9E9E9E">（無填寫）</td><td class=t style="color:#9E9E9E">（無填寫）</td></tr>
</table>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>

<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>5</td><td class=h>主要品牌庫存週數（Jul'26 pipeline）</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5a｜MI3.0 庫存週數趨勢（2025-07 ～ 2026-07，Unit: week）</td></tr></table>
<!--CHART2: PNG 由 v2 截圖或 generate_chart 產出後替換 src-->
<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART2_PNG_URL}}" width=590 height=328 alt="主要品牌庫存週數趨勢（MI3.0），Unit: week" style="display:block;border:0;max-width:100%"></td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>趨勢判讀：</b>Dell 全期 24–25w 高位盤整未見去化；ASUS 自 2026-04 起由 18w 急升至 24w，與 Dell 收斂並進入紅燈區；HP 於 2026-01 觸底 8.5w 後強力反彈至 14w；Samsung 自 10w 回升至 15w，兩者升勢皆入黃燈觀察區。</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5b｜Jul'26 庫存燈號</td></tr></table>
<table width=100% cellpadding=0 cellspacing=0>
<tr><th width=60>燈號</th><th width=90>品牌</th><th width=64>週數</th><th>判讀</th></tr>
<tr><td class=ce style="color:#C00000">&#9679; <b>紅燈</b></td><td class=t><b>Dell</b></td><td><b>25w</b></td><td class=t>庫存偏高，高位盤整未去化</td></tr>
<tr bgcolor=#EFF5FB><td class=ce style="color:#C00000">&#9679; <b>紅燈</b></td><td class=t><b>ASUS</b></td><td><b>24w</b></td><td class=t>庫存偏高，近月急升與 Dell 收斂</td></tr>
<tr><td class=ce style="color:#946C00"><font color=#BF8F00>&#9679;</font> <b>黃燈</b></td><td class=t><b>Samsung</b></td><td><b>15w</b></td><td class=t>庫存升高，需謹慎觀察</td></tr>
<tr bgcolor=#EFF5FB><td class=ce style="color:#946C00"><font color=#BF8F00>&#9679;</font> <b>黃燈</b></td><td class=t><b>HP</b></td><td><b>14w</b></td><td class=t>庫存升高（自 8.5w 低點反彈），需謹慎觀察</td></tr>
</table>
</td></tr></table></td></tr>

<tr><td class=f>本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 Jul'26 填報。<br>本信為 Outlook 表格降級版；完整圖表版請開啟隨信 .html 附件。互動篩選請至 Databricks Dashboard。</td></tr>
</table>
</td></tr></table>
</body></html>


'''

In [ ]:
%pip install -U anthropic httpx
dbutils.library.restartPython()

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime

# ── 設定 ──
TARGET_TABLE = "micenter.mi3_datahub_prod.g_sales_brief_mail_report"
APP_CODES = ["TV", "NB", "MNT"]

# ── 來源表最新月份（三表 period 交集，再取各 app 最新）──
source_latest = spark.sql("""
    WITH numeric_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
        WHERE status = 'submitted'
    ),
    brand_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM micenter.mi3_datahub_prod.g_sales_brief_brand_summary
    ),
    app_periods AS (
        SELECT DISTINCT app_code, fill_year, fill_month
        FROM micenter.mi3_datahub_prod.g_sales_brief_app_summary
    ),
    common AS (
        SELECT n.app_code, n.fill_year, n.fill_month
        FROM numeric_periods n
        INNER JOIN brand_periods b
            ON n.app_code = b.app_code AND n.fill_year = b.fill_year AND n.fill_month = b.fill_month
        INNER JOIN app_periods a
            ON n.app_code = a.app_code AND n.fill_year = a.fill_year AND n.fill_month = a.fill_month
    )
    SELECT app_code,
           MAX(fill_year * 100 + fill_month) AS source_latest_period
    FROM common
    GROUP BY app_code
""").collect()

source_map = {row["app_code"]: row["source_latest_period"] for row in source_latest}
print("來源表最新完整月份（三表交集）:", source_map)

# ── 全部重新產出 ──
apps_to_process = []
for app in APP_CODES:
    src = source_map.get(app)
    if src:
        apps_to_process.append({
            "app_code": app,
            "fill_year": src // 100,
            "fill_month": src % 100
        })
        print(f"✅ {app}: 來源={src} → 產出")
    else:
        print(f"⏭️ {app}: 無來源數據 → 跳過")

print(f"\n本次需產出: {[a['app_code'] for a in apps_to_process]}")

In [ ]:
from collections import defaultdict
import calendar
import json

# ═══ APP 特殊設定（不影響 TV/NB 通用性）═══
APP_CONFIG = {
    "TV": {
        "y_prev_override": {
            "Samsung": 33.8,
            "Hisense": 30.0,
            "TCL": 30.0,
            "TPV": 6.2,
        },
    },
    "MNT": {
        "brand_order": ["Dell", "HP", "Samsung", "ASUS"],
        "brand_display": {"Asus": "ASUS"},  # DB raw → 顯示名稱
        "y25_yoy_override": {"Dell": -5, "HP": -23, "Samsung": -6, "ASUS": 27},
        # Major 合計列的 Y25 YoY%（y_prev_yoy）寫死值；不設或設 None＝維持以加總值計算
        "major_y25_yoy_override": -7,
        # inv_thresholds 格式：(red_high, yellow_high) 或 (red_high, yellow_high, red_low)
        #   2 元組＝只判斷上緣紅燈；3 元組＝低於 red_low 也是紅燈（過低／斷料風險）
        "inv_thresholds": {
            "ASUS": (24, 20),
            "_default": (20, 12),
        },
    },
    "NB": {
        # NB 庫存水位遠低於 MNT/TV：
        #   >= 10w 紅燈（偏高）／>= 5w 黃燈／3w~5w 綠燈／< 3w 紅燈（過低）
        "inv_thresholds": {
            "_default": (10, 5, 3),
        },
    }
}

def _normalize_brand(raw_brand_name, app_code):
    """從前綴品牌名萃取並正規化（如 MNT_Asus → ASUS）"""
    brand = raw_brand_name.split("_", 1)[1] if "_" in raw_brand_name else raw_brand_name
    return APP_CONFIG.get(app_code, {}).get("brand_display", {}).get(brand, brand)

# ═══ 庫存燈號判斷（附件版 Section 5b 與 mail_body Section 5b 共用）═══
def _get_inv_thresholds(app_code, brand):
    """回傳 (red_high, yellow_high, red_low)；red_low 為 None 代表不判斷過低。"""
    cfg = APP_CONFIG.get(app_code, {}).get("inv_thresholds", {})
    th = cfg.get(brand, cfg.get("_default", (20, 12)))
    red_high = th[0]
    yellow_high = th[1]
    red_low = th[2] if len(th) > 2 else None
    return red_high, yellow_high, red_low


def get_inv_lamp(app_code, brand, weeks):
    """依 app/brand 閾值回傳 (color, lamp_txt, desc)。
    判斷順序：偏高紅燈 → 過低紅燈 → 黃燈 → 綠燈。"""
    red_high, yellow_high, red_low = _get_inv_thresholds(app_code, brand)
    if weeks >= red_high:
        return "#C00000", "紅燈", "庫存偏高"
    if red_low is not None and weeks < red_low:
        return "#C00000", "紅燈", "庫存過低，留意斷料風險"
    if weeks >= yellow_high:
        return "#BF8F00", "黃燈", "升高需觀察"
    return "#2E7D32", "綠燈", "庫存健康"


def get_inv_note(app_code):
    """依 app 閾值產生 Section 5b 註解文字（HTML 用，'<' 已跳脫為 &lt;）。"""
    cfg = APP_CONFIG.get(app_code, {}).get("inv_thresholds", {})
    red_high, yellow_high, red_low = _get_inv_thresholds(app_code, "_default")
    if red_low is not None:
        rule = f"≥ {red_high}w 紅燈（偏高）、≥ {yellow_high}w 黃燈、{red_low}w ~ {yellow_high}w 綠燈、&lt; {red_low}w 紅燈（過低）"
    else:
        rule = f"≥ {red_high}w 紅燈、≥ {yellow_high}w 黃燈、&lt; {yellow_high}w 綠燈"
    note = f"註：庫存週數基於業務填報數據，燈號閾值：{rule}。"
    ov = [b for b in cfg.keys() if b != "_default"]
    if ov:
        detail = "；".join(
            f"{b} ≥{cfg[b][0]}w 紅燈／≥{cfg[b][1]}w 黃燈" for b in ov
        )
        note += f"（品牌例外：{detail}）"
    return note

# ═══ Major 合計列（Section 1a / 1b 的第一列，附件版與 mail 版共用）═══
def _sum_opt(values):
    """加總並忽略 None；全部為 None 時回傳 None。"""
    vals = [v for v in values if v is not None]
    return sum(vals) if vals else None


def build_major_yearly(brands_ordered, yearly_data, app_code=""):
    """回傳 Major 合計列，結構與 yearly_data 的單一品牌 dict 相同。
    各金額欄為所有品牌加總；YoY% / Hit% 一律以加總後的值重算
    （不沿用個別品牌的 y25_yoy_override，避免加總列與明細列邏輯衝突）。
    例外：若 APP_CONFIG[app_code] 有設 major_y25_yoy_override，
    則 Y25 YoY%（y_prev_yoy）直接以該寫死值覆蓋。"""
    def g(key):
        return _sum_opt([yearly_data.get(b, {}).get(key) for b in brands_ordered])

    y_prev2 = g("y_prev2")
    y_prev  = g("y_prev")
    y_pre   = g("y_cur_pre")
    y_cur   = g("y_cur")
    ytd     = g("ytd")

    def _yoy(cur, base):
        return ((cur - base) / base * 100) if (cur is not None and base) else None

    row = {
        "y_prev2": y_prev2,
        "y_prev": y_prev,
        "y_cur_pre": y_pre,
        "y_cur": y_cur,
        "y_prev_yoy": _yoy(y_prev, y_prev2),
        "y_cur_pre_yoy": _yoy(y_pre, y_prev),
        "y_cur_yoy": _yoy(y_cur, y_prev),
        "ytd": ytd,
        "hit_pct": (ytd / y_cur * 100) if (ytd is not None and y_cur) else None,
        "inv_weeks": None,
    }

    major_override = APP_CONFIG.get(app_code, {}).get("major_y25_yoy_override")
    if major_override is not None:
        row["y_prev_yoy"] = major_override

    return row


def build_major_quarterly(brands_ordered, quarterly_data, quarters):
    """回傳 {(year, quarter): 合計值}，僅涵蓋 quarters 清單內的季度。"""
    return {
        key: _sum_opt([quarterly_data.get(b, {}).get(key) for b in brands_ordered])
        for key in quarters
    }


def major_label(brands_ordered):
    """Major 列的顯示名稱，例如 6 個品牌 → 'Major 6'。"""
    return f"Major {len(brands_ordered)}"


# ═══ LLM Helper（尚未初始化時用 fallback）═══
_LLM_READY = True

def call_claude(messages, max_tokens=2048, thinking=False):
    """Placeholder - will be overridden by LLM init cell."""
    return ""

def generate_insight(section_name, data_summary):
    """Call LLM to generate insight paragraph. Falls back to empty if LLM not ready."""
    if not _LLM_READY:
        return "（AI 分析尚未啟用）"
    prompt = f"""你是 AUO 顯示器業務分析師。請根據以下數據，用繁體中文撰寫一段中性、簡潔的高階主管摘要（100-200字）。
要求：點出各品牌主要趨勢、變化原因、風險提示，本區重點是資訊正確的解讀，不要用過多解釋。 不要markdown符號，直接輸出純文字。

Section: {section_name}
數據:
{data_summary}"""
    return call_claude([{"role": "user", "content": prompt}], max_tokens=500, thinking=False) or "（AI 分析產生失敗）"

# ═══════════ 品牌排序 ═══════════
def get_brand_order(app_code, fill_year, fill_month):
    """Return list of brand names ordered by yearly BP (descending).
    使用 target_month = fill_month 取得當月的年度 BP 值。"""
    # APP_CONFIG 固定排序優先
    if app_code in APP_CONFIG and "brand_order" in APP_CONFIG[app_code]:
        return APP_CONFIG[app_code]["brand_order"]
    rows = spark.sql(f"""
        SELECT brand_code, brand_name,
               MAX(CASE WHEN category='panel_buy' AND metric='yearly_bp_current' THEN value END) AS bp
        FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
          AND target_month = {fill_month}
          AND status = 'submitted'
        GROUP BY brand_code, brand_name
        ORDER BY bp DESC
    """).collect()
    return [_normalize_brand(r["brand_name"], app_code) for r in rows]


# ═══════════ 年度數據 (Section 1a) ═══════════
def get_yearly_data(app_code, fill_year, fill_month):
    """Get Y24, Y25, Y26(Pre), Y26(Cur), YTD per brand for Section 1a.
    使用 target_year/target_month 正確取得各時點的年度 BP 值。"""
    # Y26 Cur = 當月提交的 yearly_bp, target_month = fill_month (當月的年度預測)
    cur = spark.sql(f"""
        SELECT brand_code, brand_name,
               MAX(CASE WHEN category='panel_buy' AND metric='yearly_bp_current' THEN value END) AS y26_cur,
               MAX(CASE WHEN category='inventory' AND metric='monthly' THEN value END) AS inv_weeks
        FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
          AND target_month = {fill_month}
          AND status = 'submitted'
        GROUP BY brand_code, brand_name
    """).collect()

    # Y26 Pre = 前一個 target period 的 yearly_bp（依 target_month 定義前版，再取最新 submission）
    prev_target_year, prev_target_month = (fill_year, fill_month - 1) if fill_month > 1 else (fill_year - 1, 12)
    prev = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}'
              AND category = 'panel_buy' AND metric = 'yearly_bp_current'
              AND target_year = {prev_target_year} AND target_month = {prev_target_month}
              AND status = 'submitted'
        )
        SELECT brand_code, brand_name, value AS y26_pre FROM ranked WHERE rn = 1
    """).collect()
    prev_map = {r["brand_code"]: r.asDict() for r in prev}

    # Y_prev = fill_year-1 的年度 BP（target_month=12，年底最終值），取最新 submission
    y_prev_year = fill_year - 1
    y_prev = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND target_year = {y_prev_year} AND target_month = 12
              AND category = 'panel_buy' AND metric = 'yearly_bp_current'
              AND status = 'submitted'
        )
        SELECT brand_code, brand_name, value AS y_prev FROM ranked WHERE rn = 1
    """).collect()
    y_prev_map = {r["brand_code"]: r.asDict() for r in y_prev}

    # Y_prev2 = fill_year-2 的年度 BP（target_month=12，年底最終值），取最新 submission
    y_prev2_year = fill_year - 2
    y_prev2 = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND target_year = {y_prev2_year} AND target_month = 12
              AND category = 'panel_buy' AND metric = 'yearly_bp_current'
              AND status = 'submitted'
        )
        SELECT brand_code, brand_name, value AS y_prev2 FROM ranked WHERE rn = 1
    """).collect()
    y_prev2_map = {r["brand_code"]: r.asDict() for r in y_prev2}

    # YTD = 加總 target_month 1~(fill_month-1) 的月度 panel_buy
    # 只計算已完成的過去月份，不含當月（當月為最新填報點，尚未結算）
    # 每個 target_month 取最新提交的值
    ytd_rows = spark.sql(f"""
        WITH ranked AS (
            SELECT brand_code, brand_name, target_month, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
              AND target_year = {fill_year} AND target_month < {fill_month}
        )
        SELECT brand_code, brand_name, SUM(value) AS ytd
        FROM ranked WHERE rn = 1
        GROUP BY brand_code, brand_name
    """).collect()
    ytd_map = {r["brand_code"]: float(r["ytd"]) if r["ytd"] is not None else None for r in ytd_rows}

    result = {}
    for r in cur:
        bc = r["brand_code"]
        bn = r["brand_name"]
        brand = _normalize_brand(bn, app_code)
        y26c = float(r["y26_cur"]) if r["y26_cur"] is not None else None
        y26p = float(prev_map.get(bc, {}).get("y26_pre")) if prev_map.get(bc, {}).get("y26_pre") is not None else None
        y_pv = float(y_prev_map.get(bc, {}).get("y_prev")) if y_prev_map.get(bc, {}).get("y_prev") is not None else None
        y_p2v = float(y_prev2_map.get(bc, {}).get("y_prev2")) if y_prev2_map.get(bc, {}).get("y_prev2") is not None else None
        ytd = ytd_map.get(bc)
        result[brand] = {
            "y_prev2": y_p2v, "y_prev": y_pv, "y_cur_pre": y26p, "y_cur": y26c,
            "y_prev_yoy": ((y_pv - y_p2v) / y_p2v * 100) if (y_pv and y_p2v) else None,
            "y_cur_pre_yoy": ((y26p - y_pv) / y_pv * 100) if (y26p and y_pv) else None,
            "y_cur_yoy": ((y26c - y_pv) / y_pv * 100) if (y26c and y_pv) else None,
            "ytd": ytd,
            "hit_pct": (ytd / y26c * 100) if (ytd and y26c and y26c > 0) else None,
            "inv_weeks": float(r["inv_weeks"]) if r["inv_weeks"] is not None else None,
        }
    # Apply y_prev baseline override（分析師提供的前年 baseline，覆蓋 DB 查詢值並重算相關 YoY）
    if app_code in APP_CONFIG and "y_prev_override" in APP_CONFIG[app_code]:
        overrides = APP_CONFIG[app_code]["y_prev_override"]
        for brand in result:
            if brand in overrides:
                y_pv = overrides[brand]
                result[brand]["y_prev"] = y_pv
                y_p2v = result[brand]["y_prev2"]
                y26p = result[brand]["y_cur_pre"]
                y26c = result[brand]["y_cur"]
                result[brand]["y_prev_yoy"] = ((y_pv - y_p2v) / y_p2v * 100) if (y_pv and y_p2v) else None
                result[brand]["y_cur_pre_yoy"] = ((y26p - y_pv) / y_pv * 100) if (y26p and y_pv) else None
                result[brand]["y_cur_yoy"] = ((y26c - y_pv) / y_pv * 100) if (y26c and y_pv) else None

    # Apply Y_prev YoY override（分析師手動提供 YoY 數字，直接覆寫計算值）
    if app_code in APP_CONFIG and "y25_yoy_override" in APP_CONFIG[app_code]:
        overrides = APP_CONFIG[app_code]["y25_yoy_override"]
        for brand in result:
            if brand in overrides:
                result[brand]["y_prev_yoy"] = overrides[brand]
    return result


# ═══════════ 季度數據 (Section 1b) ═══════════
def get_quarterly_data(app_code, fill_year, fill_month):
    """Get quarterly panel buy per brand.
    使用 target_year/target_month 正確分配季度，每個 target_month 取最新提交值。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
              AND target_year >= {fill_year - 1}
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
    """).collect()

    # 依 target_month 分配季度並加總
    brand_quarters = defaultdict(dict)  # brand -> {(year, q): value}
    temp = defaultdict(lambda: defaultdict(float))  # brand -> (year, q) -> sum
    for r in rows:
        bn = _normalize_brand(r["brand_name"], app_code)
        ty, tm = r["target_year"], r["target_month"]
        q = (tm - 1) // 3 + 1
        val = float(r["value"]) if r["value"] else 0
        temp[bn][(ty, q)] += val

    for bn in temp:
        for key, total in temp[bn].items():
            brand_quarters[bn][key] = total

    return dict(brand_quarters)


# ═══════════ 月度趨勢 (Section 2a SVG + 2b MoM) ═══════════
def get_monthly_series(app_code, fill_year, fill_month, n_months=18):
    """Return monthly panel_buy for SVG chart & MoM table.
    使用 target_year/target_month 作為時間軸，每個 target_month 取最新提交值。
    包含當月 submission 的未來預測月份（供 MoM 表使用）。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'panel_buy' AND metric = 'monthly'
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
        ORDER BY target_year, target_month
    """).collect()

    brand_series = defaultdict(list)
    for r in rows:
        bn = _normalize_brand(r["brand_name"], app_code)
        brand_series[bn].append({
            "year": r["target_year"], "month": r["target_month"],
            "value": float(r["value"]) if r["value"] is not None else None
        })
    for brand in brand_series:
        brand_series[brand] = brand_series[brand][-n_months:]
    return dict(brand_series)


# ═══════════ 庫存週數歷史趨勢 (Section 5a SVG) ═══════════
def get_inventory_series(app_code, fill_year, fill_month, n_months=13):
    """Return monthly inventory weeks per brand (last n_months).
    使用 target_year/target_month 作為時間軸，每個 target_month 取最新提交值。"""
    rows = spark.sql(f"""
        WITH ranked AS (
            SELECT target_year, target_month, brand_code, brand_name, value,
                   ROW_NUMBER() OVER (
                       PARTITION BY brand_code, target_year, target_month
                       ORDER BY fill_year DESC, fill_month DESC, submitted_at DESC, submission_id DESC
                   ) AS rn
            FROM micenter.mi3_datahub_prod.g_sales_brief_numeric_entry
            WHERE app_code = '{app_code}' AND status = 'submitted'
              AND category = 'inventory' AND metric = 'monthly'
              AND (target_year * 100 + target_month) <= {fill_year * 100 + fill_month}
        )
        SELECT target_year, target_month, brand_code, brand_name, value
        FROM ranked WHERE rn = 1
        ORDER BY target_year, target_month
    """).collect()

    brand_series = defaultdict(list)
    for r in rows:
        bn = _normalize_brand(r["brand_name"], app_code)
        if r["value"] is not None:
            brand_series[bn].append({
                "year": r["target_year"], "month": r["target_month"],
                "value": float(r["value"])
            })
    for brand in brand_series:
        brand_series[brand] = brand_series[brand][-n_months:]
    return dict(brand_series)


# ═══════════ 摘要數據 (Section 3 & 4) ═══════════
def get_summary_data(app_code, fill_year, fill_month):
    """Get app-level and brand-level AI summaries."""
    app_row = spark.sql(f"""
        SELECT * FROM micenter.mi3_datahub_prod.g_sales_brief_app_summary
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
    """).collect()
    brand_rows = spark.sql(f"""
        SELECT * FROM micenter.mi3_datahub_prod.g_sales_brief_brand_summary
        WHERE app_code = '{app_code}' AND fill_year = {fill_year} AND fill_month = {fill_month}
    """).collect()
    return app_row[0].asDict() if app_row else None, brand_rows


# ═══════════ 圖表中間結構（供 SVG 與 chart_data JSON 同源共用）═══════════
CHART_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9b59b6", "#e74c3c"]

def extract_chart_intermediate(brands_ordered, monthly_series, inv_series, fill_year, fill_month):
    """Extract chart data as intermediate dict shared by SVG (3a) and chart_data JSON (3c).
    Returns dict with 'chart1' (monthly panel buy) and 'chart2' (inventory weeks)."""

    # ── Chart 1: 品牌月度面板採購趨勢 ──
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods1 = sorted(all_periods)[-15:]  # last 15 months (same as SVG)

    # x_labels: first shows "YY-Mon", subsequent within same year just "Mon", forecast months append "(F)"
    x_labels1 = []
    forecast_from_index = None
    for i, (yr, mo) in enumerate(periods1):
        is_forecast = (yr * 100 + mo) > (fill_year * 100 + fill_month)
        if is_forecast and forecast_from_index is None:
            forecast_from_index = i
        mon_abbr = calendar.month_abbr[mo]
        if i == 0 or yr != periods1[i - 1][0]:
            label = f"{yr % 100}-{mon_abbr}"
        else:
            label = mon_abbr
        if is_forecast:
            label += "(F)"
        x_labels1.append(label)

    # series data per brand (aligned to periods1)
    series1 = []
    for bi, brand in enumerate(brands_ordered[:4]):
        period_val = {}
        for entry in monthly_series.get(brand, []):
            period_val[(entry["year"], entry["month"])] = entry["value"]
        data = [period_val.get(p) for p in periods1]
        series1.append({
            "name": brand,
            "color": CHART_COLORS[bi % len(CHART_COLORS)],
            "data": data
        })

    chart1 = {
        "x_labels": x_labels1,
        "forecast_from_index": forecast_from_index,
        "series": series1
    }

    # ── Chart 2: 主要品牌庫存週數趨勢 ──
    all_periods2 = set()
    for brand in brands_ordered:
        for entry in inv_series.get(brand, []):
            all_periods2.add((entry["year"], entry["month"]))
    periods2 = sorted(all_periods2)[-13:]  # last 13 months (same as inventory SVG)

    x_labels2 = []
    for i, (yr, mo) in enumerate(periods2):
        if i == 0 or yr != periods2[i - 1][0]:
            x_labels2.append(f"{yr % 100}-{mo:02d}")
        else:
            x_labels2.append(f"{mo:02d}")

    series2 = []
    for bi, brand in enumerate(brands_ordered[:4]):
        period_val = {}
        for entry in inv_series.get(brand, []):
            period_val[(entry["year"], entry["month"])] = entry["value"]
        data = [period_val.get(p) for p in periods2]
        series2.append({
            "name": brand,
            "color": CHART_COLORS[bi % len(CHART_COLORS)],
            "data": data
        })

    chart2 = {
        "x_labels": x_labels2,
        "forecast_from_index": None,  # inventory has no forecast segment
        "series": series2
    }

    return {"chart1": chart1, "chart2": chart2}


print("✅ 資料整理函式已定義")

In [ ]:
import httpx
import time as _time

FOUNDRY_API_KEY = dbutils.secrets.get(scope="llm-api-keys", key="ms-foundry")
FOUNDRY_BASE_URL = "https://micenter.services.ai.azure.com/anthropic/v1/messages"
MODEL_SONNET = "claude-sonnet-5"

def call_claude(messages, max_tokens=2048, thinking=False):
    """Call Claude via Azure Foundry with retry."""
    for attempt in range(3):
        try:
            body = {"model": MODEL_SONNET, "max_tokens": max_tokens, "messages": messages}
            if not thinking:
                body["thinking"] = {"type": "disabled"}
            resp = httpx.post(
                FOUNDRY_BASE_URL,
                headers={"x-api-key": FOUNDRY_API_KEY, "anthropic-version": "2023-06-01", "content-type": "application/json"},
                json=body, timeout=120.0,
            )
            resp.raise_for_status()
            data = resp.json()
            text_parts = [block["text"] for block in data.get("content", []) if block.get("type") == "text"]
            return "\n".join(text_parts) if text_parts else ""
        except Exception as e:
            if attempt < 2:
                _time.sleep(10 * (attempt + 1))
            else:
                print(f"  \u26a0\ufe0f LLM call failed: {e}")
                return ""

globals()['_LLM_READY'] = True
print("\u2705 LLM \u5df2\u521d\u59cb\u5316\uff0cAI Insight \u5df2\u555f\u7528")

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 3: HTML 產出函式（完整 web 版，含 SVG 图表 + AI Insight）
# ════════════════════════════════════════════════════════════

def _fmt_num(v, decimals=1):
    if v is None: return "-"
    return f"{v:.{decimals}f}"

def _fmt_pct(v):
    if v is None: return "-"
    return f"{v:+.0f}%" if v != 0 else "0%"

def _tag_pct(v):
    """Return tag-styled percentage HTML."""
    if v is None: return "-"
    txt = f"{v:+.0f}%" if v != 0 else "0%"
    if v > 0:
        return f'<span class="tag-pos">{txt}</span>'
    elif v < 0:
        return f'<span class="tag-neg">{txt}</span>'
    return f'<span class="tag-neu">{txt}</span>'

def _month_label(year, month):
    return f"{year % 100}-{calendar.month_abbr[month]}"


# ═══ CSS ═══
HTML_CSS = """
<style>
*{box-sizing:border-box;margin:0;padding:0;}
body{font-family:'Segoe UI','Noto Sans TC','Microsoft JhengHei',Arial,sans-serif;background:#F3F4F6;color:#1A1A2E;font-size:14px;}
.container{max-width:960px;margin:0 auto;padding:16px;}
.header{background:linear-gradient(135deg,#1F4E79,#2E75B6);color:#fff;border-radius:12px;padding:26px 30px;margin-bottom:20px;}
.header h1{font-size:25px;font-weight:700;margin-bottom:8px;}
.header-meta{display:flex;flex-wrap:wrap;gap:12px;font-size:13px;opacity:0.92;}
.header-meta span{background:rgba(255,255,255,0.15);padding:4px 12px;border-radius:20px;}
.section-card{background:#fff;border-radius:12px;border:1px solid #C8D8E8;padding:22px 24px;margin-bottom:20px;}
.section-title{display:flex;align-items:center;gap:12px;margin-bottom:16px;}
.section-badge{width:28px;height:28px;border-radius:50%;background:#1F4E79;color:#fff;font-size:13px;font-weight:700;display:flex;align-items:center;justify-content:center;flex-shrink:0;}
.section-title h2{font-size:17px;font-weight:700;color:#1A1A2E;}
.sub-title{font-size:14px;font-weight:700;color:#1F4E79;margin:18px 0 10px;}
.chart-wrap{position:relative;padding-bottom:56%;height:0;overflow:hidden;border-radius:8px;border:1px solid #E0E0E0;background:#FCFCFB;}
.insight-box{background:#F5F7FA;border-left:4px solid #1F4E79;padding:12px 16px;border-radius:0 6px 6px 0;margin-top:14px;font-size:13px;line-height:1.8;color:#1A1A2E;}
table{width:100%;border-collapse:collapse;font-size:13px;}
th{background:#1F4E79;color:#fff;padding:8px 9px;text-align:center;font-weight:600;white-space:nowrap;}
td{padding:7px 9px;border-bottom:1px solid #DCE6F0;}
tr:nth-child(even) td{background:#EFF5FB;}
tr:nth-child(odd) td{background:#fff;}
.td-num{text-align:right;font-variant-numeric:tabular-nums;}
.td-txt{text-align:left;}
.tbl-scroll{overflow-x:auto;}
.tag-pos{display:inline-block;background:#E8F5E9;color:#2E7D32;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neg{display:inline-block;background:#FDECEC;color:#C00000;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.tag-neu{display:inline-block;background:#FFF8E1;color:#946C00;border-radius:10px;padding:1px 8px;font-size:12px;white-space:nowrap;}
.lamp{display:inline-block;width:11px;height:11px;border-radius:50%;vertical-align:middle;margin-right:6px;}
.note{font-size:11px;color:#8A8A96;margin-top:8px;line-height:1.6;}
.footer{text-align:center;font-size:12px;color:#9E9E9E;padding:14px 0 24px;line-height:1.7;}
.facet-label{width:84px;background:#EDF2F7 !important;color:#1F4E79;font-weight:700;vertical-align:top;}
</style>
"""


# ═══ Section 1a: 年度 Panel Buy 總表 ═══
def _build_section1a(brands_ordered, yearly_data, fill_year, fill_month, app_code=""):
    def _row(label, d, is_major=False):
        hit = d.get("hit_pct")
        bg = ' style="background:#EDF2F7 !important;"' if is_major else ''
        return f"""          <tr><td class="td-txt"{bg}><strong>{label}</strong></td>
            <td class="td-num"{bg}>{_fmt_num(d.get('y_prev'))}</td><td class="td-num"{bg}>{_fmt_num(d.get('y_cur_pre'))}</td>
            <td class="td-num"{bg}><strong>{_fmt_num(d.get('y_cur'))}</strong></td>
            <td class="td-num"{bg}>{_tag_pct(d.get('y_prev_yoy'))}</td>
            <td class="td-num"{bg}>{_tag_pct(d.get('y_cur_pre_yoy'))}</td>
            <td class="td-num"{bg}>{_tag_pct(d.get('y_cur_yoy'))}</td>
            <td class="td-num"{bg}>{_fmt_num(d.get('ytd'))}</td>
            <td class="td-num"{bg}>{f'{hit:.0f}%' if hit is not None else '-'}</td></tr>\n"""

    # Major 合計列排在最上方
    rows_html = _row(major_label(brands_ordered),
                     build_major_yearly(brands_ordered, yearly_data, app_code),
                     is_major=True)
    for brand in brands_ordered:
        rows_html += _row(brand, yearly_data.get(brand, {}))

    return f"""    <div class="sub-title">1a｜年度 Panel Buy 總表（Unit: M pcs）</div>
    <div class="tbl-scroll"><table>
      <thead><tr><th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th><th>Y25 YoY%</th><th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th><th>YTD (1~{fill_month - 1}M)</th><th>Hit%</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>"""


# ═══ Section 1b: 季度 Panel Buy 明細 ═══
def _build_section1b(brands_ordered, quarterly_data, fill_year, fill_month):
    cur_q = (fill_month - 1) // 3 + 1
    # 季度欄位：前一年 Q1~Q4 + 當年 Q1~cur_q
    quarters = []
    for q in range(1, 5):
        quarters.append((fill_year - 1, q))
    for q in range(1, cur_q + 1):
        quarters.append((fill_year, q))

    # Header
    header = "<th>Brand</th>"
    for yr, q in quarters:
        label = f"{yr % 100} {q}Q"
        if yr == fill_year and q == cur_q:
            label += " (Cur)"
        header += f"<th>{label}</th>"
    for q in range(1, cur_q + 1):
        header += f"<th>{fill_year % 100} {q}Q QoQ</th>"

    def _row(label, bq, is_major=False):
        bg = ' style="background:#EDF2F7 !important;"' if is_major else ''
        row = f'<td class="td-txt"{bg}><strong>{label}</strong></td>'
        qvals = []
        for yr, q in quarters:
            v = bq.get((yr, q))
            qvals.append(v)
            if yr == fill_year and q == cur_q:
                row += f'<td class="td-num"{bg}><strong>{_fmt_num(v)}</strong></td>'
            else:
                row += f'<td class="td-num"{bg}>{_fmt_num(v)}</td>'
        # 當年各季 QoQ
        for qi in range(len(quarters) - cur_q, len(quarters)):
            cur_v = qvals[qi]
            prev_v = qvals[qi - 1] if qi > 0 else None
            qoq = ((cur_v - prev_v) / prev_v * 100) if (cur_v and prev_v and prev_v != 0) else None
            row += f'<td class="td-num"{bg}>{_tag_pct(qoq)}</td>'
        return f"          <tr>{row}</tr>\n"

    # Major 合計列排在最上方
    rows_html = _row(major_label(brands_ordered),
                     build_major_quarterly(brands_ordered, quarterly_data, quarters),
                     is_major=True)
    for brand in brands_ordered:
        rows_html += _row(brand, quarterly_data.get(brand, {}))

    return f"""    <div class="sub-title">1b｜季度 Panel Buy 明細（Unit: M pcs）</div>
    <div class="tbl-scroll"><table>
      <thead><tr>{header}</tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
    <div class="note">註：QoQ 為對前一季變化率（{fill_year} 1Q QoQ 以 {fill_year-1} 4Q 為基期）；(Cur)＝本次填報；Major {len(brands_ordered)} 為各品牌加總，QoQ 以加總值計算。</div>"""


# ═══ Section 2a: SVG 折線圖 ═══
def _build_svg_chart(brands_ordered, monthly_series, fill_year, fill_month):
    """Generate SVG line chart for monthly panel buy trend."""
    COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9b59b6", "#e74c3c"]
    W, H = 720, 400
    PAD_L, PAD_R, PAD_T, PAD_B = 46, 92, 40, 56
    CHART_W = W - PAD_L - PAD_R
    CHART_H = H - PAD_T - PAD_B

    # Collect all data points across brands
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods = sorted(all_periods)[-15:]  # last 15 months
    if not periods:
        return "<!-- No data for SVG chart -->"

    # Determine value range
    all_vals = []
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            if (entry["year"], entry["month"]) in set(periods) and entry["value"]:
                all_vals.append(entry["value"])
    if not all_vals:
        return "<!-- No values for SVG chart -->"
    v_max = max(all_vals) * 1.15
    v_min = 0

    def x_pos(idx):
        return PAD_L + (idx / max(len(periods) - 1, 1)) * CHART_W

    def y_pos(val):
        if val is None or v_max == v_min:
            return PAD_T + CHART_H
        return PAD_T + CHART_H - ((val - v_min) / (v_max - v_min)) * CHART_H

    # Build SVG
    svg_parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 {W} {H}" role="img" aria-label="品牌月度面板採購趨勢" style="position:absolute;top:0;left:0;display:block;">']
    svg_parts.append(f'<text x="{W/2}" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#1A1A2E">品牌月度面板採購趨勢（MI3.0）</text>')
    svg_parts.append(f'<text x="{PAD_L}" y="24" text-anchor="start" font-size="11" fill="#898781">Unit: M pcs</text>')

    # Y-axis grid lines
    n_grid = 4
    for i in range(n_grid + 1):
        yv = v_min + (v_max - v_min) * i / n_grid
        yp = y_pos(yv)
        dash = ' stroke-dasharray="3 3"' if i > 0 else ''
        svg_parts.append(f'<line x1="{PAD_L}" y1="{yp:.1f}" x2="{W-PAD_R}" y2="{yp:.1f}" stroke="#E1E0D9"{dash} stroke-width="1"/>')
        svg_parts.append(f'<text x="{PAD_L-6}" y="{yp+4:.1f}" text-anchor="end" font-size="11" fill="#898781">{yv:.1f}</text>')

    # X-axis labels (every 3rd)
    for i, (yr, mo) in enumerate(periods):
        if i % 3 == 0 or i == len(periods) - 1:
            xp = x_pos(i)
            label = f"{yr % 100}-{calendar.month_abbr[mo]}"
            svg_parts.append(f'<text x="{xp:.1f}" y="{H-PAD_B+18}" text-anchor="middle" font-size="11" fill="#898781">{label}</text>')

    # Plot lines per brand
    period_set = periods
    for bi, brand in enumerate(brands_ordered[:6]):
        color = COLORS[bi % len(COLORS)]
        series = monthly_series.get(brand, [])
        pts = []
        for entry in series:
            key = (entry["year"], entry["month"])
            if key in set(period_set):
                idx = period_set.index(key)
                if entry["value"] is not None:
                    pts.append((x_pos(idx), y_pos(entry["value"]), entry["value"]))
        if len(pts) < 2:
            continue
        points_str = " ".join(f"{x:.1f},{y:.1f}" for x, y, _ in pts)
        svg_parts.append(f'<polyline points="{points_str}" fill="none" stroke="{color}" stroke-width="2" stroke-linejoin="round"/>')
        # End marker
        lx, ly, lv = pts[-1]
        svg_parts.append(f'<circle cx="{lx:.1f}" cy="{ly:.1f}" r="3.5" fill="{color}" stroke="#FCFCFB" stroke-width="2"/>')
        svg_parts.append(f'<text x="{lx+8:.1f}" y="{ly+4:.1f}" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">{brand} {lv:.1f}</text>')

    # Legend
    leg_y = H - 14
    leg_x = 86
    for bi, brand in enumerate(brands_ordered[:6]):
        color = COLORS[bi % len(COLORS)]
        svg_parts.append(f'<line x1="{leg_x}" y1="{leg_y}" x2="{leg_x+18}" y2="{leg_y}" stroke="{color}" stroke-width="3"/>')
        svg_parts.append(f'<text x="{leg_x+24}" y="{leg_y+4}" font-size="11" fill="#52514E">{brand}</text>')
        leg_x += 90

    svg_parts.append('</svg>')
    return '\n'.join(svg_parts)


# ═══ Section 2b: MoM% 變化表 ═══
def _build_section2b(brands_ordered, monthly_series, fill_year, fill_month):
    """Build MoM% change table."""
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods = sorted(all_periods)[-9:]

    # Header
    header = "<th>MoM</th>"
    for yr, mo in periods:
        header += f"<th>{_month_label(yr, mo)}</th>"

    # Major N sums
    major_sums = {}
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            key = (entry["year"], entry["month"])
            major_sums[key] = major_sums.get(key, 0) + (entry["value"] or 0)

    def _mom_row(label, series_dict, periods, bold=False):
        bld = "<strong>" if bold else ""
        bld_e = "</strong>" if bold else ""
        cells = f'<td class="td-txt">{bld}{label}{bld_e}</td>'
        prev_val = None
        for p in periods:
            val = series_dict.get(p)
            if val is not None and prev_val is not None and prev_val != 0:
                pct = (val - prev_val) / prev_val * 100
                cells += f'<td class="td-num">{_tag_pct(pct)}</td>'
            else:
                cells += '<td class="td-num">-</td>'
            prev_val = val
        return cells

    rows_html = f'          <tr>{_mom_row(f"Major {len(brands_ordered)}", major_sums, periods, bold=True)}</tr>\n'
    for brand in brands_ordered:
        bd = {(e["year"], e["month"]): e["value"] for e in monthly_series.get(brand, [])}
        rows_html += f'          <tr>{_mom_row(brand, bd, periods, bold=True)}</tr>\n'

    return f"""    <div class="sub-title">2b｜MoM% 變化表（近 9 個月）</div>
    <div class="tbl-scroll"><table>
      <thead><tr>{header}</tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
    <div class="note">註：MoM% 基於各月填報之 Panel Buy 預測總量變化。</div>"""


# ═══ Section 3: 業務訊息整體摘要 ═══
def _build_section3(app_summary):
    if not app_summary:
        return ""
    fields = [
        ("整體市況", app_summary.get("summary_futureOutlook") or "（無填寫）"),
        ("產品技術", app_summary.get("summary_techProduct") or "（無填寫）"),
        ("價格相關", app_summary.get("summary_pricing") or "（無填寫）"),
        ("其他焦點", app_summary.get("summary_otherInfo") or "（無填寫）"),
    ]
    rows = ""
    for label, content in fields:
        rows += f'        <tr><td class="facet-label">{label}</td><td class="td-txt">{content}</td></tr>\n'

    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">3</div><h2>業務訊息整體摘要</h2></div>
    <table><tbody>
{rows}    </tbody></table>
  </div>"""


# ═══ Section 4: 各品牌市場訊息矩陣 ═══
def _build_section4(brands_ordered, brand_rows, app_code=""):
    # brand_code in DB may differ in case (e.g. "Asus" vs display "ASUS")
    display_map = APP_CONFIG.get(app_code, {}).get("brand_display", {})
    brand_map = {display_map.get(r["brand_code"], r["brand_code"]): r.asDict() for r in brand_rows}
    rows_html = ""
    for brand in brands_ordered:
        r = brand_map.get(brand, {})
        outlook = r.get("summary_futureOutlook") or '<span style="color:#999">（無填寫）</span>'
        tech = r.get("summary_techProduct") or '<span style="color:#999">（無填寫）</span>'
        price = r.get("summary_pricing") or '<span style="color:#999">（無填寫）</span>'
        other = r.get("summary_otherInfo") or '<span style="color:#999">（無填寫）</span>'
        rows_html += f'          <tr><td class="td-txt"><strong>{brand}</strong></td><td class="td-txt">{outlook}</td><td class="td-txt">{tech}</td><td class="td-txt">{price}</td><td class="td-txt">{other}</td></tr>\n'

    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">4</div><h2>各品牌市場訊息矩陣</h2></div>
    <div class="tbl-scroll"><table>
      <thead><tr><th style="width:64px;">品牌</th><th style="width:26%;">市場展望</th><th style="width:26%;">產品技術</th><th style="width:23%;">價格</th><th>其他資訊</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table></div>
  </div>"""


# ═══ Section 5a: 庫存週數趨勢 SVG ═══
def _build_inv_svg_chart(brands_ordered, inv_series, fill_year, fill_month):
    """Generate SVG line chart for inventory weeks trend."""
    COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9b59b6", "#e74c3c"]
    W, H = 720, 400
    PAD_L, PAD_R, PAD_T, PAD_B = 46, 92, 40, 56
    CHART_W = W - PAD_L - PAD_R
    CHART_H = H - PAD_T - PAD_B

    # Collect all periods
    all_periods = set()
    for brand in brands_ordered:
        for entry in inv_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods = sorted(all_periods)[-13:]  # last 13 months
    if not periods:
        return "<!-- No data for inventory SVG chart -->"

    # Value range
    all_vals = []
    for brand in brands_ordered:
        for entry in inv_series.get(brand, []):
            if (entry["year"], entry["month"]) in set(periods) and entry["value"]:
                all_vals.append(entry["value"])
    if not all_vals:
        return "<!-- No values for inventory SVG chart -->"
    v_max = max(max(all_vals) * 1.1, 26)  # at least 26w for context
    v_min = 0

    def x_pos(idx):
        return PAD_L + (idx / max(len(periods) - 1, 1)) * CHART_W

    def y_pos(val):
        if val is None or v_max == v_min:
            return PAD_T + CHART_H
        return PAD_T + CHART_H - ((val - v_min) / (v_max - v_min)) * CHART_H

    svg_parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="100%" height="100%" viewBox="0 0 {W} {H}" role="img" aria-label="主要品牌庫存週數趨勢" style="position:absolute;top:0;left:0;display:block;">']
    svg_parts.append(f'<text x="{W/2}" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#1A1A2E">主要品牌庫存週數趨勢（MI3.0）</text>')
    svg_parts.append(f'<text x="{PAD_L}" y="24" text-anchor="start" font-size="11" fill="#898781">Unit: week</text>')

    # Y-axis grid (0, 5, 10, 15, 20, 25)
    for yv in range(0, int(v_max) + 1, 5):
        yp = y_pos(yv)
        dash = ' stroke-dasharray="3 3"' if yv > 0 else ''
        svg_parts.append(f'<line x1="{PAD_L}" y1="{yp:.1f}" x2="{W-PAD_R}" y2="{yp:.1f}" stroke="#E1E0D9"{dash} stroke-width="1"/>')
        svg_parts.append(f'<text x="{PAD_L-6}" y="{yp+4:.1f}" text-anchor="end" font-size="11" fill="#898781">{yv}</text>')

    # X-axis labels
    for i, (yr, mo) in enumerate(periods):
        if i % 3 == 0 or i == len(periods) - 1:
            xp = x_pos(i)
            svg_parts.append(f'<text x="{xp:.1f}" y="{H-PAD_B+18}" text-anchor="middle" font-size="11" fill="#898781">{yr % 100}-{calendar.month_abbr[mo]}</text>')

    # Plot lines
    period_set = periods
    for bi, brand in enumerate(brands_ordered[:6]):
        color = COLORS[bi % len(COLORS)]
        series = inv_series.get(brand, [])
        pts = []
        for entry in series:
            key = (entry["year"], entry["month"])
            if key in set(period_set):
                idx = period_set.index(key)
                if entry["value"] is not None:
                    pts.append((x_pos(idx), y_pos(entry["value"]), entry["value"]))
        if len(pts) < 2:
            continue
        points_str = " ".join(f"{x:.1f},{y:.1f}" for x, y, _ in pts)
        svg_parts.append(f'<polyline points="{points_str}" fill="none" stroke="{color}" stroke-width="2" stroke-linejoin="round"/>')
        lx, ly, lv = pts[-1]
        svg_parts.append(f'<circle cx="{lx:.1f}" cy="{ly:.1f}" r="3.5" fill="{color}" stroke="#FCFCFB" stroke-width="2"/>')
        svg_parts.append(f'<text x="{lx+8:.1f}" y="{ly+4:.1f}" text-anchor="start" font-size="11" font-weight="600" fill="#1A1A2E">{brand} {int(lv)}</text>')

    # Legend
    leg_y = H - 14
    leg_x = 86
    for bi, brand in enumerate(brands_ordered[:6]):
        color = COLORS[bi % len(COLORS)]
        svg_parts.append(f'<line x1="{leg_x}" y1="{leg_y}" x2="{leg_x+18}" y2="{leg_y}" stroke="{color}" stroke-width="3"/>')
        svg_parts.append(f'<text x="{leg_x+24}" y="{leg_y+4}" font-size="11" fill="#52514E">{brand}</text>')
        leg_x += 90

    svg_parts.append('</svg>')
    return '\n'.join(svg_parts)


# ═══ Section 5: 庫存週數（5a SVG + 5b 燈號表）═══
def _build_section5(brands_ordered, yearly_data, inv_series, fill_year, fill_month, app_code=""):
    month_abbr = calendar.month_abbr[fill_month]

    # 5a: SVG chart
    svg_chart = _build_inv_svg_chart(brands_ordered, inv_series, fill_year, fill_month)

    # 5b: 燈號表
    rows_html = ""
    for brand in brands_ordered:
        d = yearly_data.get(brand, {})
        weeks = d.get("inv_weeks")
        if weeks is None:
            continue
        color, lamp_txt, desc = get_inv_lamp(app_code, brand, weeks)
        rows_html += f'          <tr><td style="text-align:center;"><span class="lamp" style="background:{color};"></span><strong style="color:{color};">{lamp_txt}</strong></td><td class="td-txt"><strong>{brand}</strong></td><td class="td-num"><strong>{int(weeks)}w</strong></td><td class="td-txt">{desc}</td></tr>\n'

    return f"""  <div class="section-card">
    <div class="section-title"><div class="section-badge">5</div><h2>主要品牌庫存週數（{month_abbr}'{fill_year % 100} pipeline）</h2></div>
    <div class="sub-title">5a｜MI3.0 庫存週數趨勢（Unit: week）</div>
    <div class="chart-wrap">
{svg_chart}
    </div>
    <div class="sub-title">5b｜{month_abbr}'{fill_year % 100} 庫存燈號</div>
    <table>
      <thead><tr><th style="width:64px;">燈號</th><th style="width:110px;">品牌</th><th style="width:90px;">週數</th><th class="td-txt">判讀</th></tr></thead>
      <tbody>
{rows_html}      </tbody>
    </table>
    <div class="note">{get_inv_note(app_code)}</div>
  </div>"""


# ═══════════ 組裝完整 HTML ═══════════
def generate_html(app_code, fill_year, fill_month):
    """Generate complete HTML report matching template v2 (web, with SVG)."""
    month_abbr = calendar.month_abbr[fill_month]
    title = f"{month_abbr}'{fill_year % 100} {app_code} Sales Brief"

    # ── 取數據 ──
    print(f"  → 取品牌排序...")
    brands_ordered = get_brand_order(app_code, fill_year, fill_month)
    print(f"  → 取年度數據...")
    yearly_data = get_yearly_data(app_code, fill_year, fill_month)
    print(f"  → 取季度數據...")
    quarterly_data = get_quarterly_data(app_code, fill_year, fill_month)
    print(f"  → 取月度趨勢...")
    monthly_series = get_monthly_series(app_code, fill_year, fill_month)
    print(f"  → 取摘要數據...")
    app_summary, brand_rows = get_summary_data(app_code, fill_year, fill_month)

    # ── 組裝 Section 1 ──
    s1a = _build_section1a(brands_ordered, yearly_data, fill_year, fill_month, app_code)
    s1b = _build_section1b(brands_ordered, quarterly_data, fill_year, fill_month)

    # AI Insights for Section 1
    print(f"  → 產生 AI Insight (Section 1)...")
    yearly_summary = "; ".join([f"{b}: Y25={yearly_data.get(b,{}).get('y_prev')}, Y26(Cur)={yearly_data.get(b,{}).get('y_cur')}, YoY={yearly_data.get(b,{}).get('y_cur_yoy'):.0f}%" for b in brands_ordered if yearly_data.get(b,{}).get('y_cur_yoy') is not None])
    insight_1a = generate_insight("年度 Panel Buy 分析", yearly_summary)
    cur_q_for_prompt = (fill_month - 1) // 3 + 1
    quarterly_summary = f"【重要約束】表格資料僅到 {fill_year} {cur_q_for_prompt}Q 為止，請只分析有數字的季度，絕對不要提及或推測之後的季度。\n"
    quarterly_summary += "; ".join([f"{b}: Q data={quarterly_data.get(b, {})}" for b in brands_ordered[:4]])
    insight_1b = generate_insight("季度 Panel Buy 分析", quarterly_summary)

    # ── 組裝 Section 2 ──
    print(f"  → 產生 SVG 圖表...")
    svg_chart = _build_svg_chart(brands_ordered, monthly_series, fill_year, fill_month)
    s2b = _build_section2b(brands_ordered, monthly_series, fill_year, fill_month)

    # AI Insight for Section 2
    print(f"  → 產生 AI Insight (Section 2)...")
    mom_summary = "; ".join([f"{b}: last 3 months trend = {[e.get('value') for e in monthly_series.get(b, [])[-3:]]}" for b in brands_ordered[:4]])
    insight_2 = generate_insight("月度採購趨勢分析", mom_summary)

    # ── 組裝 Section 3~5 ──
    s3 = _build_section3(app_summary)
    s4 = _build_section4(brands_ordered, brand_rows, app_code)
    print(f"  → 取庫存趨勢數據...")
    inv_series = get_inventory_series(app_code, fill_year, fill_month)
    s5 = _build_section5(brands_ordered, yearly_data, inv_series, fill_year, fill_month, app_code)

    # ── 組裝完整 HTML ──
    html = f"""<!DOCTYPE html>
<html lang="zh-Hant">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>{title}</title>
{HTML_CSS}
</head>
<body>
<div class="container">

  <!-- Header -->
  <div class="header">
    <h1>{title}</h1>
    <div class="header-meta">
      <span>&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測</span>
      <span>&#128202; 資料來源：MI DataHub / MI3.0</span>
      <span>&#128337; 數據截至：{month_abbr}'{fill_year % 100} 填報</span>
    </div>
  </div>

  <!-- 頂部提示 -->
  <div style="background:#FFF8E1;border:1px solid #FFD54F;border-radius:8px;padding:12px 18px;margin-bottom:18px;font-size:13px;color:#5D4037;line-height:1.7;">
    &#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。
  </div>

  <!-- Section 1 -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">1</div><h2>品牌年度／季度 Panel buy 分析</h2></div>
{s1a}
    <div class="insight-box">&#128204; <strong>AI 年度分析：</strong>{insight_1a}</div>
{s1b}
    <div class="insight-box">&#128204; <strong>AI 季度分析：</strong>{insight_1b}</div>
  </div>

  <!-- Section 2 -->
  <div class="section-card">
    <div class="section-title"><div class="section-badge">2</div><h2>品牌月度面板採購</h2></div>
    <div class="sub-title">2a｜MI3.0 月度採購趨勢（Unit: M pcs）</div>
    <div class="chart-wrap">
{svg_chart}
    </div>
    <div class="insight-box">&#128204; <strong>趨勢判讀：</strong>{insight_2}</div>
{s2b}
  </div>

{s3}

{s4}

{s5}

  <!-- Footer -->
  <div class="footer">
    本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 {month_abbr}'{fill_year % 100} 填報；完整互動圖表請至 Databricks Dashboard 查看。
  </div>

</div>
</body>
</html>"""
    return html

# ════════════════════════════════════════════════════════════
# Step 3b: generate_mail_body — Email 內文 HTML
# ════════════════════════════════════════════════════════════

# Email-specific helpers (class-based, matching MAIL_BODY_TEMPLATE CSS)
def _mail_tag(v):
    """Return email-safe percentage tag."""
    if v is None:
        return "-"
    txt = f"{v:+.0f}%" if v != 0 else "0%"
    if v > 0:
        return f'<span class=p>{txt}</span>'
    elif v < 0:
        return f'<span class=n>{txt}</span>'
    return f'<span class=u>{txt}</span>'


def _mail_section_start(num, title):
    return f'''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFFFFF style="border:1px solid #C8D8E8"><tr><td class=w>
<table cellpadding=0 cellspacing=0><tr><td class=b width=24 height=24>{num}</td><td class=h>{title}</td></tr></table>'''


def _mail_section_end():
    return '</td></tr></table></td></tr>\n<tr><td class=s height=14></td></tr>'


def generate_mail_body(app_code, fill_year, fill_month,
                       brands_ordered, yearly_data, quarterly_data,
                       monthly_series, app_summary, brand_rows,
                       yearly_data_full, inv_series,
                       insight_1a, insight_1b, insight_2):
    """Generate Email-compatible HTML (table-based, 640px).
    Keeps {{CHART1_PNG_URL}} and {{CHART2_PNG_URL}} placeholders for downstream PNG replacement.
    Returns minified HTML string."""
    import re as _re
    month_abbr = calendar.month_abbr[fill_month]
    title = f"{month_abbr}'{fill_year % 100} {app_code} Sales Brief"

    # ── CSS head (same as MAIL_BODY_TEMPLATE) ──
    css = '''<style>
body{margin:0;background:#F3F4F6}
table{border-collapse:collapse}
td,th{font-family:'Microsoft JhengHei','Noto Sans TC',sans-serif;font-size:12px}
th{background:#1F4E79;color:#FFFFFF;padding:5px 3px;font-size:11px}
td{padding:5px 4px;border-bottom:1px solid #DCE6F0;text-align:right}
.t{text-align:left}
.ce{text-align:center}
.w{padding:18px 22px;border:0;text-align:left}
.s{padding:0;border:0;font-size:0;line-height:0}
.b{background:#1F4E79;color:#FFFFFF;font-weight:700;text-align:center;border-radius:50%;padding:0;border:0}
.h{font-size:16px;font-weight:700;padding:0 0 0 10px;border:0;text-align:left}
.st{color:#1F4E79;font-weight:700;font-size:13px;padding:14px 0 7px;border:0;text-align:left}
.i{background:#F5F7FA;padding:10px 14px;line-height:1.8;border:0;text-align:left}
.no{font-size:11px;color:#8A8A96;padding:6px 0 0;border:0;text-align:left;line-height:1.6}
.fl{background:#EDF2F7;color:#1F4E79;font-weight:700;text-align:left;vertical-align:top;width:76px}
.p{background:#E8F5E9;color:#2E7D32;border-radius:8px;padding:0 5px;font-size:11px}
.n{background:#FDECEC;color:#C00000;border-radius:8px;padding:0 5px;font-size:11px}
.u{background:#FFF8E1;color:#946C00;border-radius:8px;padding:0 5px;font-size:11px}
.f{font-size:11px;color:#9E9E9E;text-align:center;line-height:1.7;padding:12px 0 20px;border:0}
</style>'''

    parts = []
    parts.append(f'''<!DOCTYPE html>
<html xmlns:o="urn:schemas-microsoft-com:office:office" lang=zh-Hant><head><meta charset=UTF-8><title>{title}</title>
<!--[if mso]><noscript><xml><o:OfficeDocumentSettings><o:PixelsPerInch>96</o:PixelsPerInch></o:OfficeDocumentSettings></xml></noscript><![endif]-->
{css}</head>
<body bgcolor=#F3F4F6>
<table width=100% cellpadding=0 cellspacing=0 bgcolor=#F3F4F6><tr><td class=s align=left>
<table width=640 cellpadding=0 cellspacing=0 align=left>
<tr><td class=s height=16></td></tr>''')

    # Header
    parts.append(f'''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#1F4E79><tr><td class=w style="padding:22px 24px">
<span style="font-size:22px;font-weight:700;color:#FFFFFF">{title}</span><br>
<span style="font-size:12px;color:#CFE0F0;line-height:2">&#128197; 業務資訊蒐集｜Panel Buy / 市場訊息 / 庫存觀測　&#128202; 資料來源：MI DataHub / MI3.0　&#128337; 數據截至：{month_abbr}\'{fill_year % 100} 填報</span>
</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>''')

    # Top notice
    parts.append('''<tr><td class=s><table width=100% cellpadding=0 cellspacing=0 bgcolor=#FFF8E1 style="border:1px solid #FFD54F"><tr><td class=w style="padding:10px 16px;color:#5D4037;line-height:1.7">&#128196; 完整圖表版報告已隨信附上（.html 附件），請以瀏覽器開啟以檢視互動圖表。</td></tr></table></td></tr>
<tr><td class=s height=14></td></tr>''')

    # ── Section 1: 年度／季度 Panel Buy ──
    parts.append(_mail_section_start(1, '品牌年度／季度 Panel buy 分析'))
    # 1a table
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1a｜年度 Panel Buy 總表（Unit: M pcs）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    parts.append(f'<tr><th>Brand</th><th>Y25</th><th>Y26 (Pre)</th><th>Y26 (Cur)</th><th>Y25 YoY%</th><th>Y26 (Pre) YoY%</th><th>Y26 YoY%</th><th>YTD (1~{fill_month-1}M)</th><th>Hit%</th></tr>')
    _rows_1a = [(major_label(brands_ordered), build_major_yearly(brands_ordered, yearly_data, app_code), True)]
    _rows_1a += [(b, yearly_data.get(b, {}), False) for b in brands_ordered]
    for ri, (_label, d, _is_major) in enumerate(_rows_1a):
        bg = ' bgcolor=#EDF2F7' if _is_major else (' bgcolor=#EFF5FB' if ri % 2 == 1 else '')
        y25 = _fmt_num(d.get('y_prev'))
        y26p = _fmt_num(d.get('y_cur_pre'))
        y26c = _fmt_num(d.get('y_cur'))
        ytd = _fmt_num(d.get('ytd'))
        hit = f"{d.get('hit_pct'):.0f}%" if d.get('hit_pct') is not None else '-'
        parts.append(f'<tr{bg}><td class=t><b>{_label}</b></td><td>{y25}</td><td>{y26p}</td><td><b>{y26c}</b></td><td>{_mail_tag(d.get("y_prev_yoy"))}</td><td>{_mail_tag(d.get("y_cur_pre_yoy"))}</td><td>{_mail_tag(d.get("y_cur_yoy"))}</td><td>{ytd}</td><td>{hit}</td></tr>')
    parts.append('</table>')
    # 1a insight
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 年度分析：</b>{insight_1a or ""}</td></tr></table>')

    # 1b quarterly table
    cur_q = (fill_month - 1) // 3 + 1
    quarters = [(fill_year - 1, q) for q in range(1, 5)] + [(fill_year, q) for q in range(1, cur_q + 1)]
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>1b｜季度 Panel Buy 明細（Unit: M pcs）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    hdr = '<th>Brand</th>' + ''.join(f'<th>{yr%100} {q}Q{" (Cur)" if yr==fill_year and q==cur_q else ""}</th>' for yr, q in quarters)
    hdr += ''.join(f'<th>{fill_year%100} {q}Q QoQ</th>' for q in range(1, cur_q + 1))
    parts.append(f'<tr>{hdr}</tr>')
    _rows_1b = [(major_label(brands_ordered),
                 build_major_quarterly(brands_ordered, quarterly_data, quarters), True)]
    _rows_1b += [(b, quarterly_data.get(b, {}), False) for b in brands_ordered]
    for ri, (_label, bq, _is_major) in enumerate(_rows_1b):
        bg = ' bgcolor=#EDF2F7' if _is_major else (' bgcolor=#EFF5FB' if ri % 2 == 1 else '')
        cells = f'<td class=t><b>{_label}</b></td>'
        qvals = []
        for yr, q in quarters:
            v = bq.get((yr, q))
            qvals.append(v)
            if yr == fill_year and q == cur_q:
                cells += f'<td><b>{_fmt_num(v)}</b></td>'
            else:
                cells += f'<td>{_fmt_num(v)}</td>'
        for qi in range(len(quarters) - cur_q, len(quarters)):
            cv = qvals[qi]
            pv = qvals[qi - 1] if qi > 0 else None
            qoq = ((cv - pv) / pv * 100) if (cv and pv and pv != 0) else None
            cells += f'<td>{_mail_tag(qoq)}</td>'
        parts.append(f'<tr{bg}>{cells}</tr>')
    parts.append('</table>')
    # 1b insight
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>AI 季度分析：</b>{insight_1b or ""}</td></tr></table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：QoQ 為對前一季變化率（{fill_year} 1Q QoQ 以 {fill_year-1} 4Q 為基期）；(Cur)＝本次填報；Major {len(brands_ordered)} 為各品牌加總，QoQ 以加總值計算。</td></tr></table>')
    parts.append(_mail_section_end())

    # ── Section 2: 月度採購 ──
    parts.append(_mail_section_start(2, '品牌月度面板採購'))
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2a｜MI3.0 月度採購趨勢（Unit: M pcs）</td></tr></table>')
    # Chart 1 placeholder
    parts.append('<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART1_PNG_URL}}" width=590 height=328 alt="四品牌月度面板採購趨勢（MI3.0），Unit: M pcs" style="display:block;border:0;max-width:100%"></td></tr></table>')
    # Insight 2
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=8 colspan=2></td></tr><tr><td class=s width=4 bgcolor=#1F4E79></td><td class=i>&#128204; <b>趨勢判讀：</b>{insight_2 or ""}</td></tr></table>')

    # 2b MoM table
    all_periods = set()
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            all_periods.add((entry["year"], entry["month"]))
    periods = sorted(all_periods)[-9:]
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>2b｜MoM% 變化表（近 9 個月）</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    mom_hdr = '<th>MoM</th>' + ''.join(f'<th>{_month_label(yr, mo)}</th>' for yr, mo in periods)
    parts.append(f'<tr>{mom_hdr}</tr>')

    # Major N sum row
    major_sums = {}
    for brand in brands_ordered:
        for entry in monthly_series.get(brand, []):
            k = (entry["year"], entry["month"])
            major_sums[k] = major_sums.get(k, 0) + (entry["value"] or 0)

    def _mom_cells(label, series_dict, periods_list, bold=False):
        b = '<b>' if bold else ''
        be = '</b>' if bold else ''
        c = f'<td class=t>{b}{label}{be}</td>'
        prev_val = None
        for p in periods_list:
            val = series_dict.get(p)
            if val is not None and prev_val is not None and prev_val != 0:
                pct = (val - prev_val) / prev_val * 100
                c += f'<td>{_mail_tag(pct)}</td>'
            else:
                c += '<td>-</td>'
            prev_val = val
        return c

    parts.append(f'<tr>{_mom_cells(f"Major {len(brands_ordered)}", major_sums, periods, bold=True)}</tr>')
    for ri, brand in enumerate(brands_ordered):
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 0 else ''
        bd = {(e["year"], e["month"]): e["value"] for e in monthly_series.get(brand, [])}
        parts.append(f'<tr{bg}>{_mom_cells(brand, bd, periods, bold=True)}</tr>')
    parts.append('</table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>註：MoM% 基於各月填報之 Panel Buy 預測總量變化。</td></tr></table>')
    parts.append(_mail_section_end())

    # ── Section 3: 業務訊息整體摘要 ──
    if app_summary:
        parts.append(_mail_section_start(3, '業務訊息整體摘要'))
        parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=10 colspan=2></td></tr>')
        fields = [
            ("整體市況", app_summary.get("summary_futureOutlook") or "（無填寫）"),
            ("產品技術", app_summary.get("summary_techProduct") or "（無填寫）"),
            ("價格相關", app_summary.get("summary_pricing") or "（無填寫）"),
            ("其他焦點", app_summary.get("summary_otherInfo") or "（無填寫）"),
        ]
        for fi, (label, content) in enumerate(fields):
            bg = ' bgcolor=#EFF5FB' if fi % 2 == 1 else ''
            parts.append(f'<tr{bg}><td class=fl>{label}</td><td class=t style="vertical-align:top">{content}</td></tr>')
        parts.append('</table>')
        parts.append(_mail_section_end())

    # ── Section 4: 各品牌市場訊息矩陣 ──
    display_map = APP_CONFIG.get(app_code, {}).get("brand_display", {})
    brand_map = {display_map.get(r["brand_code"], r["brand_code"]): r.asDict() for r in brand_rows}
    parts.append(_mail_section_start(4, '各品牌市場訊息矩陣'))
    parts.append('<table width=100% cellpadding=0 cellspacing=0><tr><td class=s height=10 colspan=5></td></tr>')
    parts.append('<tr><th width=44>品牌</th><th width=145>市場展望</th><th width=145>產品技術</th><th width=130>價格</th><th>其他資訊</th></tr>')
    for ri, brand in enumerate(brands_ordered):
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 1 else ''
        r = brand_map.get(brand, {})
        na = '<span style="color:#9E9E9E">（無填寫）</span>'
        outlook = r.get("summary_futureOutlook") or na
        tech = r.get("summary_techProduct") or na
        price = r.get("summary_pricing") or na
        other = r.get("summary_otherInfo") or na
        parts.append(f'<tr valign=top{bg}><td class=t><b>{brand}</b></td><td class=t>{outlook}</td><td class=t>{tech}</td><td class=t>{price}</td><td class=t>{other}</td></tr>')
    parts.append('</table>')
    parts.append(_mail_section_end())

    # ── Section 5: 庫存週數 ──
    parts.append(_mail_section_start(5, f'主要品牌庫存週數（{month_abbr}\'{fill_year % 100} pipeline）'))
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5a｜MI3.0 庫存週數趨勢（Unit: week）</td></tr></table>')
    # Chart 2 placeholder
    parts.append('<table width=100% cellpadding=0 cellspacing=0 bgcolor=#FCFCFB style="border:1px solid #E0E0E0"><tr><td class=s align=center><img src="{{CHART2_PNG_URL}}" width=590 height=328 alt="主要品牌庫存週數趨勢（MI3.0），Unit: week" style="display:block;border:0;max-width:100%"></td></tr></table>')
    # Inventory signal table
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=st>5b｜{month_abbr}\'{fill_year % 100} 庫存燈號</td></tr></table>')
    parts.append('<table width=100% cellpadding=0 cellspacing=0>')
    parts.append('<tr><th width=60>燈號</th><th width=90>品牌</th><th width=64>週數</th><th>判讀</th></tr>')
    for ri, brand in enumerate(brands_ordered):
        d = yearly_data.get(brand, {})
        weeks = d.get("inv_weeks")
        if weeks is None:
            continue
        bg = ' bgcolor=#EFF5FB' if ri % 2 == 1 else ''
        color, lamp_txt, desc = get_inv_lamp(app_code, brand, weeks)
        parts.append(f'<tr{bg}><td class=ce style="color:{color}">&#9679; <b>{lamp_txt}</b></td><td class=t><b>{brand}</b></td><td><b>{int(weeks)}w</b></td><td class=t>{desc}</td></tr>')
    parts.append('</table>')
    parts.append(f'<table width=100% cellpadding=0 cellspacing=0><tr><td class=no>{get_inv_note(app_code)}</td></tr></table>')
    parts.append(_mail_section_end())

    # Footer
    parts.append(f'<tr><td class=f>本報告由 MI DataHub / myAgent 自動彙整產出，數據截至 {month_abbr}\'{fill_year % 100} 填報。<br>本信為 Outlook 表格降級版；完整圖表版請開啟隨信 .html 附件。互動篩選請至 Databricks Dashboard。</td></tr>')
    parts.append('</table>\n</td></tr></table>\n</body></html>')

    raw_html = '\n'.join(parts)
    # Minify: collapse multiple whitespace/newlines into single space, strip leading/trailing
    minified = _re.sub(r'\s+', ' ', raw_html).strip()
    return minified


# ════════════════════════════════════════════════════════════
# Step 3c: generate_chart_data — 圖表數據 JSON
# ════════════════════════════════════════════════════════════

def generate_chart_data(app_code, brief_month, chart_intermediate):
    """Generate chart_data JSON string per spec 3c schema.
    chart_intermediate comes from extract_chart_intermediate()."""
    c1 = chart_intermediate["chart1"]
    c2 = chart_intermediate["chart2"]

    payload = {
        "app_name": app_code,
        "brief_month": brief_month,
        "charts": [
            {
                "id": "CHART1",
                "placeholder": "{{CHART1_PNG_URL}}",
                "chart_type": "line",
                "title": "四品牌月度面板採購趨勢（MI3.0）",
                "unit": "M pcs",
                "x_labels": c1["x_labels"],
                "forecast_from_index": c1["forecast_from_index"],
                "series": [
                    {"name": s["name"], "color": s["color"], "data": s["data"]}
                    for s in c1["series"]
                ]
            },
            {
                "id": "CHART2",
                "placeholder": "{{CHART2_PNG_URL}}",
                "chart_type": "line",
                "title": "主要品牌庫存週數趨勢（MI3.0）",
                "unit": "week",
                "x_labels": c2["x_labels"],
                "forecast_from_index": c2["forecast_from_index"],
                "series": [
                    {"name": s["name"], "color": s["color"], "data": s["data"]}
                    for s in c2["series"]
                ]
            }
        ]
    }
    return json.dumps(payload, ensure_ascii=False, separators=(',', ':'))


print("✅ HTML 產出函式已定義（完整 web 版，含 SVG + AI Insight + mail_body + chart_data）")

In [ ]:
from pyspark.sql import Row
from datetime import datetime

# ── 產出 HTML + mail_body + chart_data 並收集結果 ──
results = []

for app_info in apps_to_process:
    app_code = app_info["app_code"]
    fill_year = app_info["fill_year"]
    fill_month = app_info["fill_month"]
    brief_month = f"{fill_year}-{fill_month:02d}"

    print(f"\n{'='*60}")
    print(f"產出 {app_code} / {brief_month}")
    print(f"{'='*60}")

    try:
        # 3a: 附件版 html_content（既有）
        html_content = generate_html(app_code, fill_year, fill_month)
        print(f"  → html_content: {len(html_content)} chars")

        # ── 取數（與 generate_html 內同源，但需在外層取一次供 mail_body 使用） ──
        print(f"  → 產生 mail_body & chart_data...")
        brands_ordered = get_brand_order(app_code, fill_year, fill_month)
        yearly_data = get_yearly_data(app_code, fill_year, fill_month)
        quarterly_data = get_quarterly_data(app_code, fill_year, fill_month)
        monthly_series = get_monthly_series(app_code, fill_year, fill_month)
        app_summary, brand_rows = get_summary_data(app_code, fill_year, fill_month)
        inv_series = get_inventory_series(app_code, fill_year, fill_month)

        # AI Insights (reuse same prompts as generate_html)
        yearly_summary = "; ".join([f"{b}: Y25={yearly_data.get(b,{}).get('y25')}, Y26(Cur)={yearly_data.get(b,{}).get('y26_cur')}, YoY={yearly_data.get(b,{}).get('y26_yoy'):.0f}%" for b in brands_ordered if yearly_data.get(b,{}).get('y26_yoy') is not None])
        insight_1a = generate_insight("年度 Panel Buy 分析", yearly_summary)
        cur_q = (fill_month - 1) // 3 + 1
        quarterly_summary = f"【重要約束】表格資料僅到 {fill_year} {cur_q}Q 為止，請只分析有數字的季度，絕對不要提及或推測之後的季度。\n"
        quarterly_summary += "; ".join([f"{b}: Q data={quarterly_data.get(b, {})}" for b in brands_ordered[:4]])
        insight_1b = generate_insight("季度 Panel Buy 分析", quarterly_summary)
        mom_summary = "; ".join([f"{b}: last 3 months trend = {[e.get('value') for e in monthly_series.get(b, [])[-3:]]}" for b in brands_ordered[:4]])
        insight_2 = generate_insight("月度採購趨勢分析", mom_summary)

        # 3b: mail_body
        mail_body = generate_mail_body(
            app_code, fill_year, fill_month,
            brands_ordered, yearly_data, quarterly_data,
            monthly_series, app_summary, brand_rows,
            yearly_data, inv_series,
            insight_1a, insight_1b, insight_2
        )
        print(f"  → mail_body: {len(mail_body)} chars")

        # 3c: chart_data (同源中間結構)
        chart_intermediate = extract_chart_intermediate(
            brands_ordered, monthly_series, inv_series, fill_year, fill_month
        )
        chart_data = generate_chart_data(app_code, brief_month, chart_intermediate)
        print(f"  → chart_data: {len(chart_data)} chars")

        results.append({
            "app_name": app_code,
            "brief_month": brief_month,
            "html_content": html_content,
            "created_at": datetime.now(),
            "mail_body": mail_body,
            "chart_data": chart_data,
        })
        print(f"✅ {app_code}/{brief_month} 三欄產出成功")
    except Exception as e:
        print(f"❌ {app_code}/{brief_month} 產出失敗: {e}")
        import traceback
        traceback.print_exc()

print(f"\n── 總計產出 {len(results)} 份報告 ──")

# ── 寫入 Delta Table ──
if results:
    schema = StructType([
        StructField("app_name", StringType(), False),
        StructField("brief_month", StringType(), False),
        StructField("html_content", StringType(), False),
        StructField("created_at", TimestampType(), False),
        StructField("mail_body", StringType(), True),
        StructField("chart_data", StringType(), True),
    ])

    df_results = spark.createDataFrame([Row(**r) for r in results], schema=schema)

    # MERGE：以 (app_name, brief_month) 為 key，有就更新、沒有就新增
    df_results.createOrReplaceTempView("_sales_brief_new")

    # 確保目標表存在（首次執行時建立）
    if not spark.catalog.tableExists(TARGET_TABLE):
        df_results.write.saveAsTable(TARGET_TABLE)
        spark.sql(f"COMMENT ON TABLE {TARGET_TABLE} IS 'Sales Brief 郵件報告，by 應用/月份（html_content + mail_body + chart_data）'")
        print(f"✅ 首次建立 {TARGET_TABLE}，共 {len(results)} 筆")
    else:
        spark.sql(f"""
            MERGE INTO {TARGET_TABLE} AS t
            USING _sales_brief_new AS s
            ON t.app_name = s.app_name AND t.brief_month = s.brief_month
            WHEN MATCHED THEN UPDATE SET
                t.html_content = s.html_content,
                t.created_at = s.created_at,
                t.mail_body = s.mail_body,
                t.chart_data = s.chart_data
            WHEN NOT MATCHED THEN INSERT *
        """)
        print(f"✅ 已 MERGE 寫入 {TARGET_TABLE}，本次處理 {len(results)} 筆（歷史月份保留）")
else:
    print("⚠️ 無需產出的報告")

In [ ]:
# ════════════════════════════════════════════════════════════
# Step 5: 驗證結果 & HTML 預覽
# ════════════════════════════════════════════════════════════
import re

TABLE = "micenter.mi3_datahub_prod.g_sales_brief_mail_report"
ALL_APPS = ["TV", "NB", "MNT"]

# ── 讀取目標表三欄（全應用）──
df_check = spark.sql(f"""
    SELECT app_name, brief_month,
           LENGTH(html_content) AS html_len,
           LENGTH(mail_body) AS mail_len,
           LENGTH(chart_data) AS chart_len,
           created_at
    FROM {TABLE}
    WHERE app_name IN ('TV', 'NB', 'MNT')
    ORDER BY app_name, brief_month DESC
""")
display(df_check)

overall_pass = True
for APP in ALL_APPS:
    row = spark.sql(f"""
        SELECT html_content, mail_body, chart_data
        FROM {TABLE}
        WHERE app_name = '{APP}'
        ORDER BY brief_month DESC LIMIT 1
    """).first()

    if not row:
        print(f"❌ {APP}: 目標表無資料")
        overall_pass = False
        continue

    html_content = row["html_content"]
    mail_body = row["mail_body"]
    chart_data_str = row["chart_data"]

    print(f"\n{'='*60}")
    print(f"品質驗證關卡 — {APP}")
    print(f"{'='*60}")
    all_pass = True

    # 1️⃣ 佔位符完整性
    print("\n── 1. 佔位符完整性 ──")
    c1_count = mail_body.count("{{CHART1_PNG_URL}}") if mail_body else 0
    c2_count = mail_body.count("{{CHART2_PNG_URL}}") if mail_body else 0
    other_placeholders = re.findall(r'\{\{[^}]+\}\}', mail_body or '')
    other_placeholders = [p for p in other_placeholders if p not in ('{{CHART1_PNG_URL}}', '{{CHART2_PNG_URL}}')]
    html_placeholders = re.findall(r'\{\{[^}]+\}\}', html_content or '')

    if c1_count == 1 and c2_count == 1 and not other_placeholders and not html_placeholders:
        print("  ✅ mail_body 含 CHART1/CHART2 各一次，無殘留佔位符；html_content 無佔位符")
    else:
        all_pass = False
        print(f"  ❌ CHART1={c1_count}, CHART2={c2_count}, 其他殘留={other_placeholders}, html佔位符={html_placeholders}")

    # 2️⃣ 體積預算
    print("\n── 2. 體積預算 ──")
    mail_len = len(mail_body) if mail_body else 0
    if mail_len <= 35000:
        print(f"  ✅ mail_body = {mail_len:,} chars (≤ 35,000)")
    else:
        all_pass = False
        print(f"  ❌ mail_body = {mail_len:,} chars (超過 35,000 上限!)")

    # 3️⃣ chart JSON 可解析
    print("\n── 3. chart JSON 可解析 ──")
    try:
        cd = json.loads(chart_data_str)
        charts = cd.get("charts", [])
        chart_ok = True
        for chart in charts:
            x_len = len(chart.get("x_labels", []))
            for s in chart.get("series", []):
                if len(s.get("data", [])) != x_len:
                    chart_ok = False
                    print(f"  ❌ {chart['id']}: series '{s['name']}' data len={len(s['data'])} != x_labels len={x_len}")
        # Check chart count matches placeholders
        expected_charts = c1_count + c2_count
        if len(charts) != expected_charts:
            chart_ok = False
            print(f"  ❌ charts 數={len(charts)}, mail_body 佔位符數={expected_charts}")
        if chart_ok:
            print(f"  ✅ JSON 解析成功，{len(charts)} 組圖表，series/x_labels 長度一致")
    except Exception as e:
        all_pass = False
        print(f"  ❌ JSON 解析失敗: {e}")

    # 4️⃣ HTML 結構
    print("\n── 4. HTML 結構 ──")
    html_ends = (html_content or '').rstrip().endswith('</html>')
    mail_ends = (mail_body or '').rstrip().endswith('</html>')
    mail_no_svg = '<svg' not in (mail_body or '')
    mail_no_script = '<script' not in (mail_body or '')
    if html_ends and mail_ends and mail_no_svg and mail_no_script:
        print("  ✅ 兩版均以 </html> 結尾；mail_body 無 <svg>/<script>")
    else:
        all_pass = False
        print(f"  ❌ html_ends={html_ends}, mail_ends={mail_ends}, no_svg={mail_no_svg}, no_script={mail_no_script}")

    # 5️⃣ insight 完整性
    print("\n── 5. insight 完整性 ──")
    # Check no broken HTML fragments from empty/malformed insights
    # Valid pattern: <b>AI 年度分析：</b>content...
    # Broken patterns: empty bold tags, empty insight cells, unclosed structures
    broken_patterns = ['<b></b>', '<td class=i></td>', '<td class=i> </td>', '()</b>']
    found_broken = [p for p in broken_patterns if p in (mail_body or '')]
    if not found_broken:
        print("  ✅ insight 區塊結構完整（無殘破片段）")
    else:
        # Warning only - empty insight is acceptable if structure is intact
        print(f"  ⚠️ 可能的殘破片段: {found_broken}")

    # ── 總結 ──
    print(f"\n{'='*60}")
    if all_pass:
        print(f"✅ {APP} 全部驗證通過！三欄產出符合 v2 規格。")
    else:
        overall_pass = False
        print(f"❌ {APP} 有驗證未通過，請檢查上方錯誤。")
    print(f"{'='*60}")

print(f"\n{'━'*60}")
if overall_pass:
    print("🎉 全部應用驗證通過！")
else:
    print("⚠️ 部分應用驗證未通過，請往上檢查。")
print(f"{'━'*60}")

In [ ]:
# ── 預覽 MNT 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'MNT'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])

In [ ]:
# ── 預覽 TV 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'TV'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])

In [ ]:
# ── 預覽 NB 最新三欄（一次複製3）──
import json

row = spark.sql("""
    SELECT html_content, mail_body, chart_data
    FROM micenter.mi3_datahub_prod.g_sales_brief_mail_report
    WHERE app_name = 'NB'
    ORDER BY brief_month DESC
    LIMIT 1
""").collect()[0]

# ── 合併輸出（方便一次全部複製）──
SEP = "\n" + "═" * 80 + "\n"
output = []
output.append("═" * 80)
output.append("【1/3 html_content】")
output.append("═" * 80)
output.append(row["html_content"])
output.append(SEP)
output.append("【2/3 mail_body】")
output.append("═" * 80)
output.append(row["mail_body"])
output.append(SEP)
output.append("【3/3 chart_data】")
output.append("═" * 80)
output.append(json.dumps(json.loads(row["chart_data"]), indent=2, ensure_ascii=False))
output.append("\n" + "═" * 80)

print("\n".join(output))

# ── 瀏覽器預覽（mail_body）──
displayHTML(row["mail_body"])